# AKD-EXT Closed Loop CM1 Playground


## Closed loop workflow - CM1

A pipeline of 4 agents for end-to-end research workflows. These agents are designed to work sequentially, where each stage's output feeds into the next.

**Required env vars:** `OPENAI_API_KEY`

| Stage | Agent | Purpose |
|-------|-------|---------|
| 2 | CM1CapabilityFeasibilityMapperAgent | Assesses feasibility of a research question |
| 3 | CM1WorkflowSpecBuilderAgent | Builds experiment workflow spec from hypotheses |
| 4 | CM1ExperimentImplementationAgent | Creates implementation workspace (SLURM, configs) |
| 5 | CM1ResearchReportGeneratorAgent | Generates analysis notebooks and reports |

### Stage 1: Gap Agent Output

In [2]:
# Stage 1: Gap Agent Output

research_question = """# Environmental Stability Control via Input Sounding

## Research Question

**How does atmospheric stability in the environmental temperature profile (stable vs unstable input sounding) influence tropical cyclone intensity and structural evolution?**

---

## Problem Framing

Many idealized tropical cyclone simulations assume a single environmental thermodynamic profile, but the vertical temperature stability of the environment has a strong influence on convection organization, boundary-layer overturning, and eyewall dynamics. Most previous work has focused on surface fluxes or drag processes, with the role of background thermodynamic stability (controlled via the input sounding) rarely isolated.

Understanding whether instability-driven convection enhances storm intensification — compared to stable stratification that suppresses vertical motion — can help clarify the thermodynamic constraints on tropical cyclone development.

- **Origin:** Research design extension
- **Confidence:** Moderate
- **Rationale:** Environmental stability directly controls convective buoyancy, vertical mass flux, and eyewall convection strength — key drivers of tropical cyclone intensity.

---

## Evidence Anchor (Intra-Corpus)

- **Convective instability and tropical cyclone intensification:** Emanuel (1986, 1995)
- **Environmental thermodynamic control of convection:** Rotunno & Emanuel (1987)
- **Role of environmental stability in eyewall convection and storm structure:** Montgomery & Smith (2014)

---

## Variables / Diagnostics

- Maximum wind speed evolution
- Minimum central pressure
- Convective intensity and vertical velocity distribution
- Eyewall structure and radius of maximum wind (RMW)
- CAPE / atmospheric stability metrics derived from sounding
- Boundary layer thermodynamic structure
- Eyewall updraft magnitude and organization

---

## Causality Guardrails

- Stability changes **must only be introduced** through temperature profile modifications in the input sounding.
- Surface fluxes and drag formulations **must remain constant** to isolate thermodynamic effects.
- Interpretation should focus on structural and dynamical response before attributing causal mechanisms.

---

## Hypothesis

**Hypothesis 3.1:**
Tropical cyclone intensity will increase more rapidly under an unstable environmental temperature profile because enhanced buoyancy strengthens eyewall convection and vertical mass flux, whereas a stable temperature profile suppresses deep convection, leading to weaker storm intensification and reduced peak intensity.

---

## Why This Remains Secondary

Environmental stability interacts strongly with other thermodynamic controls such as surface fluxes, moisture availability, and drag processes. Without isolating these mechanisms first, interpretation of stability-driven intensity changes may be confounded by coupled processes.
"""

### Stage 2: Capability/Feasibility Mapper Agent

In [2]:
from akd_ext.agents.closed_loop.cm1 import (
    CM1CapabilityFeasibilityMapperAgent,
    CM1CapabilityFeasibilityMapperConfig,
)
from akd_ext.agents.closed_loop.stages.capability_feasibility_mapper import (
    CapabilityFeasibilityMapperInputSchema,
)

/Users/rsahoo/Desktop/AKD-EXT-Latest/akd-ext/.venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [3]:
# Stage 2: Capability Feasibility Mapper
mapper_agent = CM1CapabilityFeasibilityMapperAgent(CM1CapabilityFeasibilityMapperConfig())

result = await mapper_agent.arun(
    CapabilityFeasibilityMapperInputSchema(
        research_questions_md=research_question,
    )
)

print(result.report)

# Capability & Feasibility Assessment — Environmental Stability Control via Input Sounding (CM1 on Matrix)

>This report provides capability/feasibility assessments and evidence paths only. It does not make a final decision to run experiments; human approval is required.

## Step 1 — Validate Inputs

### Inputs detected
- Research question(s)/hypothesis provided: **Yes** (1 RQ; 1 hypothesis)
- CM1 configuration/namelist evidence provided: **Yes** (multiple reference case directories with `namelist.input` and in some cases `input_sounding`)
- Cluster/HPC policy documentation provided: **Yes** (Matrix / SLURM queues, processor/memory limits, pre-emption notes)

### Missing / incomplete inputs (affects confidence)
- No walltime limits per partition/QOS shown (needed to assess multi-day runs)
- No node inventory (nodes/cores) beyond queue caps (limits feasibility planning for large ensembles)
- No explicit CM1 variable list/memory model documentation (memory/storage estimates must use conf

In [4]:
# Saving the md file for testing purpose
from pathlib import Path

report_md_path = Path("result_report.md")
report_md_path.write_text(result.report, encoding="utf-8")
print(f"Saved report to: {report_md_path}")
print(result.report)

Saved report to: result_report.md
# Capability & Feasibility Assessment — Environmental Stability Control via Input Sounding (CM1 on Matrix)

>This report provides capability/feasibility assessments and evidence paths only. It does not make a final decision to run experiments; human approval is required.

## Step 1 — Validate Inputs

### Inputs detected
- Research question(s)/hypothesis provided: **Yes** (1 RQ; 1 hypothesis)
- CM1 configuration/namelist evidence provided: **Yes** (multiple reference case directories with `namelist.input` and in some cases `input_sounding`)
- Cluster/HPC policy documentation provided: **Yes** (Matrix / SLURM queues, processor/memory limits, pre-emption notes)

### Missing / incomplete inputs (affects confidence)
- No walltime limits per partition/QOS shown (needed to assess multi-day runs)
- No node inventory (nodes/cores) beyond queue caps (limits feasibility planning for large ensembles)
- No explicit CM1 variable list/memory model documentation (memo

In [5]:
# read the md file for testing purpose
from pathlib import Path

report_md_path = Path("result_report.md")
# report_md_path.write_text(report_md_content, encoding="utf-8")
report_md_content = report_md_path.read_text(encoding="utf-8")

### Stage 3: Workflow Spec Builder

In [6]:
from akd_ext.agents.closed_loop.cm1 import (
    CM1WorkflowSpecBuilderAgent,
    CM1WorkflowSpecBuilderConfig,
)
from akd_ext.agents.closed_loop.stages.workflow_spec_builder import (
    WorkflowSpecBuilderInputSchema,
)

In [7]:
# Stage 3: Workflow Spec Builder
# Takes Gap Agent output (hypotheses) + Stage 2 feasibility report
spec_agent = CM1WorkflowSpecBuilderAgent(CM1WorkflowSpecBuilderConfig())

spec_result = await spec_agent.arun(
    WorkflowSpecBuilderInputSchema(
        stage_1_hypotheses=research_question,
        stage_2_feasibility=report_md_content,
    )
)

print("SPEC:")
print("=" * 80)
print(spec_result.spec)
print("\nREASONING:")
print("=" * 80)
print(spec_result.reasoning)

SPEC:
# Metadata
workflow_id: CM1_STABILITY_TC_01
workflow_name: Environmental Stability Control via Input Sounding (Tropical Cyclone)
model: CM1
tag: stability
status: draft
generated_date: 2026-05-14
experiment_count: 5

## Sources
- run/config_files/cpm_RadConvEquil/namelist.input
- run/config_files/hurricane_3d_cpm/input_sounding
- run/config_files/hurricane_3d_cpm/namelist.input
- run/config_files/hurricane_axisymmetric/input_sounding
- run/config_files/hurricane_axisymmetric/namelist.input

# Control Definition
## Research question and hypothesis traceability
- rq_tag_or_rq_id: RQ-001
- hypothesis_id: H-001 (Hypothesis 3.1)
- scientific objective: Isolate the impact of environmental thermodynamic stability (introduced only via temperature-profile edits to `input_sounding`) on tropical cyclone (TC) intensification rate and peak intensity, while holding surface fluxes and drag constant.

## Baseline experiment
- experiment_id: EXP_stability_baseline
- control_label: baseline
- base

In [2]:
# Save the workflow spec as a markdown file
spec_md_path = Path("workflow_spec.md")
spec_md_path.write_text(spec_result.spec + "\n\n---\n\n# Reasoning\n\n" + spec_result.reasoning, encoding="utf-8")
print(f"Saved workflow spec to: {spec_md_path}")

NameError: name 'Path' is not defined

In [1]:
from pathlib import Path

workflow_spec_path = Path("workflow_spec.md")
workflow_spec_content = workflow_spec_path.read_text(encoding="utf-8")

### Stage 4 A: Experiment Implementation Agent - LLM Part to generate the JSON

In [10]:
from akd_ext.agents.closed_loop.cm1 import (
    CM1ExperimentImplementationAgent,
    CM1ExperimentImplementationConfig,
)
from akd_ext.agents.closed_loop.stages.experiment_implementation import (
    ExperimentImplementationInputSchema,
    ExperimentImplementationOutputSchema,
)

In [11]:
# Stage 4A: Experiment Implementation Planner
# GPT generates experiments, calls job_submit MCP, returns job_id + report
impl_agent = CM1ExperimentImplementationAgent(CM1ExperimentImplementationConfig())

impl_result = await impl_agent.arun(ExperimentImplementationInputSchema(stage_3_spec=workflow_spec_content))

if isinstance(impl_result, ExperimentImplementationOutputSchema):
    print(f"Job ID: {impl_result.job_id}")
    print(f"Workspace name: {impl_result.workspace_name}")
    print()
    print("REPORT:")
    print("=" * 80)
    print(impl_result.report)
else:
    print(impl_result.content)

Job ID: 01a74011-2cdd-46c6-8d12-1847095a051e
Workspace name: cm1_tc_stability_sounding

REPORT:
## CM1 Stability–TC workflow: implementation summary

**Job ID:** `01a74011-2cdd-46c6-8d12-1847095a051e`  
**Workspace:** `cm1_tc_stability_sounding`  
**Base template:** `hurricane_axisymmetric`

### 1. Experiments created (5 total)

#### EXP_stability_baseline (baseline, OK)
Edits applied:
- `namelist.input` (`param9`): enable CAPE-family diagnostics
  - `output_cape=1`, `output_cin=1`, `output_lcl=1`, `output_lfc=1`

Evidence for parameter names/groups: `run/config_files/hurricane_axisymmetric/namelist.input`

#### EXP_stability_001 (unstable, OK)
Inherits all baseline diagnostics edits plus temperature-only sounding perturbation:
- `input_sounding`: apply piecewise Δθ(z) using additive edits to `theta` column:
  - 1–3 km: linear ramp to **-2 K**
  - ≥3 km: **-2 K** constant
  - 3–12 km: additional linear ramp to **-4 K** (so total goes from -2 to -6)
  - ≥12 km: additional **-4 K** const

In [12]:
# Save the implementation report
if isinstance(impl_result, ExperimentImplementationOutputSchema):
    impl_md_path = Path("experiment_implementation.md")
    impl_md_path.write_text(impl_result.report, encoding="utf-8")
    print(f"Saved implementation report to: {impl_md_path}")
    print(f"Job ID: {impl_result.job_id}")
    print(f"Workspace name: {impl_result.workspace_name}")

Saved implementation report to: experiment_implementation.md
Job ID: 01a74011-2cdd-46c6-8d12-1847095a051e
Workspace name: cm1_tc_stability_sounding


### Stage 5: Research Report Generator

Takes the Stage 3 workflow spec + experiment IDs from Stage 4A.
The agent checks experiment status and fetches figure URLs via MCP tools,
then generates a publication-style scientific report.

**Input:** Workflow spec + experiment IDs (from Stage 4A)

**Output:** Markdown report with Abstract, Introduction, Methodology, Results, Discussion, Conclusion

**Note:** Without the MCP server, use `tools=[]` and override the prompt to skip status/figure checks.

In [13]:
# job_id= "ef389bf9-ee41-4416-805e-cfb35864f8c0"
# workspace_name = "cm1_rq001_tc_stability_sounding"

In [3]:
# Stage 5: Experiment Analysis Agent
# Polls job_status (deterministic, no LLM). If still running or failed, returns a
# free-form TextOutput and STOPS — no LLM call. If complete: fetches figure URLs,
# delegates to the generic ImageAnalyzerAgent (which batches, vision-analyzes,
# and renders Markdown), and returns the report.
job_id = "714041b2-1aca-45de-82e4-26f8452e9507"
workspace_name = "cm1_rq001_axisymm_tc_stability_sounding"

# job_id = "01a74011-2cdd-46c6-8d12-1847095a051e"
# workspace_name = "cm1_tc_stability_sounding"
from akd_ext.agents.closed_loop.cm1 import (
    CM1ExperimentAnalysisAgent,
    CM1ExperimentAnalysisConfig,
)
from akd_ext.agents.closed_loop.stages import (
    ExperimentAnalysisInputSchema,
    ExperimentAnalysisOutputSchema,
)
from IPython.display import Markdown, display

job_id = job_id
workspace_name = workspace_name
print(f"Job ID: {job_id}")
print(f"Workspace: {workspace_name}")

analysis_agent = CM1ExperimentAnalysisAgent(CM1ExperimentAnalysisConfig())

analysis_result = await analysis_agent.arun(
    ExperimentAnalysisInputSchema(
        job_id=job_id,
        workspace_name=workspace_name,
        hypothesis=research_question,
        experiment_spec=workflow_spec_content,
        skip_status_check=True,
    ),
)

if isinstance(analysis_result, ExperimentAnalysisOutputSchema):
    print()
    print(f"Status: {analysis_result.status}")
    print(f"Message: {analysis_result.message}")
    print("=" * 80)
    display(Markdown(analysis_result.markdown))
else:
    # Job not complete → status message
    print()
    print(analysis_result.content)

/Users/rsahoo/Desktop/AKD-EXT-Latest/akd-ext/.venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


Job ID: 714041b2-1aca-45de-82e4-26f8452e9507
Workspace: cm1_rq001_axisymm_tc_stability_sounding


2026-05-14 05:55:56.140 | INFO     | akd_ext.agents.closed_loop.stages.experiment_analysis:_arun:303 - [Stage5] Skipping status check for 714041b2-1aca-45de-82e4-26f8452e9507
2026-05-14 05:57:35.155 | INFO     | akd_ext.agents.closed_loop.stages.experiment_analysis:_fetch_plot_urls:293 - [Stage5] job_id=714041b2-1aca-45de-82e4-26f8452e9507 workspace='cm1_rq001_axisymm_tc_stability_sounding' figures=45 groups=['EXP_RQ001_001', 'EXP_RQ001_002', 'EXP_RQ001_baseline']
2026-05-14 05:57:35.939 | WARNING  | akd_ext.utils:fetch:55 - [download_images] unsupported format for https://s.odsi.io/wmdkg2m; skipping.
2026-05-14 06:00:00.222 | WARNING  | akd_ext.utils:fetch:55 - [download_images] unsupported format for https://s.odsi.io/umx1xhp; skipping.
2026-05-14 06:00:00.678 | WARNING  | akd_ext.utils:fetch:55 - [download_images] unsupported format for https://s.odsi.io/t6e37g2; skipping.
2026-05-14 06:02:18.595 | WARNING  | akd_ext.utils:fetch:55 - [download_images] unsupported format for https://


Status: completed
Message: Experiment Finished and Analysis is ready


# Image Analysis Report

## Context

## Hypothesis / Feasibility Report

# Environmental Stability Control via Input Sounding

## Research Question

**How does atmospheric stability in the environmental temperature profile (stable vs unstable input sounding) influence tropical cyclone intensity and structural evolution?**

---

## Problem Framing

Many idealized tropical cyclone simulations assume a single environmental thermodynamic profile, but the vertical temperature stability of the environment has a strong influence on convection organization, boundary-layer overturning, and eyewall dynamics. Most previous work has focused on surface fluxes or drag processes, with the role of background thermodynamic stability (controlled via the input sounding) rarely isolated.

Understanding whether instability-driven convection enhances storm intensification — compared to stable stratification that suppresses vertical motion — can help clarify the thermodynamic constraints on tropical cyclone development.

- **Origin:** Research design extension
- **Confidence:** Moderate
- **Rationale:** Environmental stability directly controls convective buoyancy, vertical mass flux, and eyewall convection strength — key drivers of tropical cyclone intensity.

---

## Evidence Anchor (Intra-Corpus)

- **Convective instability and tropical cyclone intensification:** Emanuel (1986, 1995)
- **Environmental thermodynamic control of convection:** Rotunno & Emanuel (1987)
- **Role of environmental stability in eyewall convection and storm structure:** Montgomery & Smith (2014)

---

## Variables / Diagnostics

- Maximum wind speed evolution
- Minimum central pressure
- Convective intensity and vertical velocity distribution
- Eyewall structure and radius of maximum wind (RMW)
- CAPE / atmospheric stability metrics derived from sounding
- Boundary layer thermodynamic structure
- Eyewall updraft magnitude and organization

---

## Causality Guardrails

- Stability changes **must only be introduced** through temperature profile modifications in the input sounding.
- Surface fluxes and drag formulations **must remain constant** to isolate thermodynamic effects.
- Interpretation should focus on structural and dynamical response before attributing causal mechanisms.

---

## Hypothesis

**Hypothesis 3.1:**
Tropical cyclone intensity will increase more rapidly under an unstable environmental temperature profile because enhanced buoyancy strengthens eyewall convection and vertical mass flux, whereas a stable temperature profile suppresses deep convection, leading to weaker storm intensification and reduced peak intensity.

---

## Why This Remains Secondary

Environmental stability interacts strongly with other thermodynamic controls such as surface fluxes, moisture availability, and drag processes. Without isolating these mechanisms first, interpretation of stability-driven intensity changes may be confounded by coupled processes.

---

## Experiment Specification

# Metadata
workflow_id: CM1_STABILITY_TC_01
workflow_name: Environmental Stability Control via Input Sounding (Tropical Cyclone)
model: CM1
tag: stability
status: draft
generated_date: 2026-05-14
experiment_count: 5

## Sources
- run/config_files/cpm_RadConvEquil/namelist.input
- run/config_files/hurricane_3d_cpm/input_sounding
- run/config_files/hurricane_3d_cpm/namelist.input
- run/config_files/hurricane_axisymmetric/input_sounding
- run/config_files/hurricane_axisymmetric/namelist.input

# Control Definition
## Research question and hypothesis traceability
- rq_tag_or_rq_id: RQ-001
- hypothesis_id: H-001 (Hypothesis 3.1)
- scientific objective: Isolate the impact of environmental thermodynamic stability (introduced only via temperature-profile edits to `input_sounding`) on tropical cyclone (TC) intensification rate and peak intensity, while holding surface fluxes and drag constant.

## Baseline experiment
- experiment_id: EXP_stability_baseline
- control_label: baseline
- baseline configuration source: `run/config_files/hurricane_axisymmetric/` using:
  - `namelist.input`
  - `input_sounding`

## Global control/guardrails applied to all experiments
- Stability perturbations are introduced only via edits to the temperature-related column of `input_sounding`.
- Moisture and wind columns in `input_sounding` are held fixed across experiments.
- Surface flux and drag settings are held fixed by not changing `&param12` (and by verifying identical `&param12` blocks between paired experiments within the same configuration family).

## Assumptions (explicit)
- The hurricane-case `input_sounding` format is interpreted as: first line gives surface reference values; subsequent lines are vertical levels where the second numeric column is the temperature-like variable (commonly potential temperature) and the third numeric column is water vapor (commonly mixing ratio), followed by wind components. This assumption must be validated by the user before applying the sounding edits.
- “Structural evolution” is assessed in two tiers: (1) axisymmetric runs for controlled sensitivity; (2) optional 3D CPM cross-checks to assess robustness under 3D convection organization.

# Experiment Matrix
One row per parameter (or file) change. Deltas are stated as instructions only.

| experiment_id | control_label | rq_tag_or_rq_id | hypothesis_id | change_target | delta_instructions (relative to that experiment’s stated base config) | what_this_tests | feasibility_flag | feasibility_constraints |
|---|---|---|---|---|---|---|---|---|
| EXP_stability_baseline | baseline | RQ-001 | H-001 | namelist:&param9:output_cape | Set `output_cape = 1` | Ensures CAPE output is available for stability diagnostics without altering physics | OK | Keep all `&param12` surface/flux settings unchanged to preserve causal guardrail |
| EXP_stability_baseline | baseline | RQ-001 | H-001 | namelist:&param9:output_cin | Set `output_cin = 1` | Ensures CIN output is available for stability diagnostics without altering physics | OK | Keep all `&param12` surface/flux settings unchanged to preserve causal guardrail |
| EXP_stability_baseline | baseline | RQ-001 | H-001 | namelist:&param9:output_lcl | Set `output_lcl = 1` | Ensures LCL output is available for stability diagnostics without altering physics | OK | Keep all `&param12` surface/flux settings unchanged to preserve causal guardrail |
| EXP_stability_baseline | baseline | RQ-001 | H-001 | namelist:&param9:output_lfc | Set `output_lfc = 1` | Ensures LFC output is available for stability diagnostics without altering physics | OK | Keep all `&param12` surface/flux settings unchanged to preserve causal guardrail |
| EXP_stability_001 |  | RQ-001 | H-001 | input_sounding:temperature_profile | Replace `input_sounding` with an “unstable” variant by applying the following edit to the temperature-like column only: apply a height-dependent perturbation Δθ(z) with Δθ=0 K for z≤1 km; linearly ramp to Δθ=-2 K at z=3 km; linearly ramp to Δθ=-6 K at z=12 km; hold Δθ=-6 K for z≥12 km. Do not change pressure/height levels, moisture column, or wind columns. | Tests whether decreased static stability (higher CAPE / weaker inhibition) accelerates intensification and increases peak intensity through stronger eyewall convection | OK | Requires `isnd=7` and a correctly formatted `input_sounding`; do not alter `&param12` |
| EXP_stability_002 |  | RQ-001 | H-001 | input_sounding:temperature_profile | Replace `input_sounding` with a “stable” variant by applying the following edit to the temperature-like column only: apply a height-dependent perturbation Δθ(z) with Δθ=0 K for z≤1 km; linearly ramp to Δθ=+2 K at z=3 km; linearly ramp to Δθ=+6 K at z=12 km; hold Δθ=+6 K for z≥12 km. Do not change pressure/height levels, moisture column, or wind columns. | Tests whether increased static stability suppresses deep convection, reducing intensification rate and peak intensity | OK | Requires `isnd=7` and a correctly formatted `input_sounding`; do not alter `&param12` |
| EXP_stability_003 |  | RQ-001 | H-001 | base_config_selection | Use `run/config_files/hurricane_3d_cpm/namelist.input` and `run/config_files/hurricane_3d_cpm/input_sounding` as the starting point (3D CPM cross-check). | Provides a 3D robustness check of the instability impact on intensification and convective structure | CONSTRAINT_DEPENDENT | High compute/walltime demand; subject to queue walltime limits and pre-emption policies |
| EXP_stability_003 |  | RQ-001 | H-001 | namelist:&param9:output_cape | Set `output_cape = 1` | Ensures CAPE output is available in the 3D cross-check | CONSTRAINT_DEPENDENT | Keep all `&param12` surface/flux settings unchanged within the 3D configuration |
| EXP_stability_003 |  | RQ-001 | H-001 | namelist:&param9:output_cin | Set `output_cin = 1` | Ensures CIN output is available in the 3D cross-check | CONSTRAINT_DEPENDENT | Keep all `&param12` surface/flux settings unchanged within the 3D configuration |
| EXP_stability_003 |  | RQ-001 | H-001 | namelist:&param9:output_lcl | Set `output_lcl = 1` | Ensures LCL output is available in the 3D cross-check | CONSTRAINT_DEPENDENT | Keep all `&param12` surface/flux settings unchanged within the 3D configuration |
| EXP_stability_003 |  | RQ-001 | H-001 | namelist:&param9:output_lfc | Set `output_lfc = 1` | Ensures LFC output is available in the 3D cross-check | CONSTRAINT_DEPENDENT | Keep all `&param12` surface/flux settings unchanged within the 3D configuration |
| EXP_stability_003 |  | RQ-001 | H-001 | input_sounding:temperature_profile | Replace `input_sounding` with an “unstable” variant using the same Δθ(z) rule as EXP_stability_001 (Δθ=0 K for z≤1 km; ramp to -2 K at 3 km; ramp to -6 K at 12 km; hold -6 K above), editing only the temperature-like column. | 3D test of whether instability-enhanced buoyancy strengthens eyewall convection and speeds intensification | CONSTRAINT_DEPENDENT | Requires `isnd=7` and a correctly formatted `input_sounding`; high compute/walltime demand |
| EXP_stability_004 |  | RQ-001 | H-001 | base_config_selection | Use `run/config_files/hurricane_3d_cpm/namelist.input` and `run/config_files/hurricane_3d_cpm/input_sounding` as the starting point (3D CPM cross-check). | Provides a 3D robustness check of the stability impact on intensification and convective structure | CONSTRAINT_DEPENDENT | High compute/walltime demand; subject to queue walltime limits and pre-emption policies |
| EXP_stability_004 |  | RQ-001 | H-001 | namelist:&param9:output_cape | Set `output_cape = 1` | Ensures CAPE output is available in the 3D cross-check | CONSTRAINT_DEPENDENT | Keep all `&param12` surface/flux settings unchanged within the 3D configuration |
| EXP_stability_004 |  | RQ-001 | H-001 | namelist:&param9:output_cin | Set `output_cin = 1` | Ensures CIN output is available in the 3D cross-check | CONSTRAINT_DEPENDENT | Keep all `&param12` surface/flux settings unchanged within the 3D configuration |
| EXP_stability_004 |  | RQ-001 | H-001 | namelist:&param9:output_lcl | Set `output_lcl = 1` | Ensures LCL output is available in the 3D cross-check | CONSTRAINT_DEPENDENT | Keep all `&param12` surface/flux settings unchanged within the 3D configuration |
| EXP_stability_004 |  | RQ-001 | H-001 | namelist:&param9:output_lfc | Set `output_lfc = 1` | Ensures LFC output is available in the 3D cross-check | CONSTRAINT_DEPENDENT | Keep all `&param12` surface/flux settings unchanged within the 3D configuration |
| EXP_stability_004 |  | RQ-001 | H-001 | input_sounding:temperature_profile | Replace `input_sounding` with a “stable” variant using the same Δθ(z) rule as EXP_stability_002 (Δθ=0 K for z≤1 km; ramp to +2 K at 3 km; ramp to +6 K at 12 km; hold +6 K above), editing only the temperature-like column. | 3D test of whether enhanced stability suppresses deep convection, reducing intensification rate and peak intensity | CONSTRAINT_DEPENDENT | Requires `isnd=7` and a correctly formatted `input_sounding`; high compute/walltime demand |

# Feasibility Notes
## Sounding-format and “temperature-only” edit integrity
- The feasibility of the core perturbation hinges on correctly identifying the temperature-like column in `input_sounding`.
- Implementation requirement: apply the Δθ(z) perturbation to only that temperature-like column, leaving (i) the moisture column, (ii) wind columns, and (iii) the level heights/pressures unchanged.
- Practical integrity check (non-executing, procedural): after editing, verify that only one numeric column differs when diffing the baseline `input_sounding` against the perturbed file.

## Guardrail enforcement: surface fluxes and drag held fixed
- No experiment in this workflow changes `&param12`.
- For the 3D cross-check pair (EXP_stability_003–EXP_stability_004), surface-flux settings must remain identical between the stable and unstable 3D runs.

## Axisymmetric vs 3D interpretation constraints
- EXP_stability_baseline, EXP_stability_001, and EXP_stability_002 use axisymmetric dynamics (`axisymm=1` in the reference case). These runs are suitable for clean sensitivity of intensification rate and mean structure but cannot represent 3D convective asymmetries.
- EXP_stability_003 and EXP_stability_004 are included as a robustness check for “structural evolution” in a 3D configuration; these are not intended to be directly intercompared against the axisymmetric baseline in a strict attribution sense.

## Diagnostics feasibility
- CAPE/CIN/LCL/LFC outputs are enabled via `&param9` toggles. This adds diagnostic output but does not modify the storm physics itself.
- If CAPE-family outputs are not desired (e.g., to reduce output volume), the stability metric can be computed offline from thermodynamic fields; this would be a workflow choice outside this design spec.

## HPC/throughput constraints for 3D cross-check
- The 3D CPM cross-check experiments are flagged CONSTRAINT_DEPENDENT due to large compute demand and sensitivity to queue walltime limits and pre-emption.
- If queue walltime limits cannot support a full-length 3D integration, the 3D cross-check should be deferred rather than modifying physics or shortening integration in a way that breaks comparability.

# Feasibility Summary
| constraint | impacted_experiments |
|---|---|
| Axisymmetric dynamics limit representation of 3D convective organization | EXP_stability_001, EXP_stability_002, EXP_stability_baseline |
| Correct identification of the temperature-like `input_sounding` column is required to satisfy “temperature-only” guardrail | EXP_stability_001, EXP_stability_002, EXP_stability_003, EXP_stability_004 |
| File-based sounding must remain enabled/valid (`isnd=7` and correctly formatted `input_sounding`) | EXP_stability_001, EXP_stability_002, EXP_stability_003, EXP_stability_004 |
| 3D CPM compute/walltime and pre-emption constraints may prevent completion on some cluster policies | EXP_stability_003, EXP_stability_004 |
| Surface flux/drag settings must remain unchanged within each paired comparison (procedural guardrail) | EXP_stability_001, EXP_stability_002, EXP_stability_003, EXP_stability_004 |

# Design Reasoning
- Hypothesis H-001 asserts a causal chain: environmental instability → stronger buoyancy/eyewall convection → faster intensification and higher peak intensity; environmental stability produces the opposite response. The cleanest direct test in CM1, given available reference cases, is to perturb only the environmental thermodynamic stability via the `input_sounding` temperature profile.
- Experiments EXP_stability_001 (unstable) and EXP_stability_002 (stable) are paired against EXP_stability_baseline while holding surface exchange settings constant by design (no `&param12` modifications).
- The Δθ(z) perturbation is specified with explicit magnitudes and a vertical structure so that the stability change is systematic and reproducible while keeping near-surface values fixed.
- 3D CPM cross-checks (EXP_stability_003–EXP_stability_004) are included to address the “structural evolution” component under 3D convection, but are flagged CONSTRAINT_DEPENDENT due to compute/walltime uncertainty. They function as robustness tests rather than strict extensions of the axisymmetric baseline.
- This workflow remains in `status: draft` pending explicit user approval of (i) the baseline choice (axisymmetric hurricane), and (ii) the magnitude/shape of the stability perturbations.

# Changelog
2026-05-14: Drafted CM1 experiment workflow for TC stability sensitivity using hurricane_axisymmetric baseline plus optional 3D cross-checks; specified temperature-only sounding perturbations and enabled CAPE-family diagnostics.

---

# Reasoning

Design choices and traceability:
- Model: CM1 only, because the feasibility artifact and reference configurations are CM1-specific.
- Baseline selection: Used the provided CM1 reference case hurricane_axisymmetric as the control because it is explicitly evidenced to use file-based sounding control (isnd=7 + input_sounding present) and is the most feasible configuration for a stability sensitivity study. No user-specified baseline was provided, so this is an explicit assumption rather than an override.
- Hypothesis → experiments:
  - H-001 requires isolating environmental stability. This is implemented with two single-factor perturbations that modify only the temperature-like column of input_sounding: one destabilizing (EXP_stability_001) and one stabilizing (EXP_stability_002).
  - A 3D robustness check is added (EXP_stability_003/004) to better align with “structural evolution” language, while keeping total experiments to 5.
- Guardrails:
  - Surface fluxes/drag held constant by not changing &param12 anywhere, and by calling out procedural verification within paired comparisons.
  - Moisture and winds in the sounding are not modified; deltas explicitly instruct editing only the temperature-like column.
Feasibility handling:
- Axisymmetric runs flagged OK; they are computationally and configuration-wise straightforward.
- 3D CPM cross-checks flagged CONSTRAINT_DEPENDENT due to compute intensity and unknown queue walltime/pre-emption impacts; included as optional but retained (not dropped) per feasibility rules.
- Sounding column semantics uncertainty is handled as an explicit assumption plus a constraint affecting all sounding-perturbation experiments; no CM1 README was used to avoid inventing undocumented column semantics.
Determinism and formatting:
- Experiment IDs deterministic: baseline first, then 001–004.
- Deltas alphabetized within cells; parameter rows for output toggles ordered alphabetically.
- Status remains draft per approval-gate constraint.

## Figures (39 total)

### 1. `ptdxk9x` — _plot_

![ptdxk9x](https://s.odsi.io/ptdxk9x)

**Caption:** RQ3 H3.1 — Intensity and structure vs environmental stability

**Description:** Composite figure (4-panel line_plot time series) titled “RQ3 H3.1 — Intensity and structure vs environmental stability”. All panels show a single blue line labeled “output”.

Top-left (Max horiz. wind): y-axis “Max horiz. wind (m/s)”. Starts near 14–15 m/s at 0 h, dips to ~12–13 m/s by ~5–10 h, then increases gradually to ~20 m/s by ~30 h and ~25–30 m/s by ~45–55 h. Rapid intensification occurs roughly 60–85 h, rising from ~25 m/s to ~70–80 m/s, with a short plateau/oscillation near ~80 m/s around ~90–100 h. After ~100 h the series is highly variable (roughly 70–85 m/s) with intermittent dips (e.g., ~62–65 m/s near ~105 h and again near ~155 h). Late period shows another increase, reaching ~88–91 m/s around ~180–185 h (apparent peak), then ending around ~85–87 m/s near 195–200 h.

Top-right (Min sfc pressure): y-axis “Min sfc pressure (hPa)”. Begins around ~1009–1010 hPa, slowly decreases to ~1000 hPa by ~40–50 h and ~996–997 hPa by ~65–70 h. A sharp drop occurs around ~70–85 h down to ~945 hPa (with continued brief decreases). After ~85 h, pressure fluctuates strongly, mostly ~935–955 hPa, with local minima around ~930 hPa (notably near ~125 h and again near ~185–190 h). Ends near ~936–938 hPa.

Bottom-left (RMW): y-axis “RMW (km)”. Starts high (~75–85 km) during 0–~15 h, then shows abrupt collapses with near-vertical excursions to very small radii (~0–2 km) between ~18–30 h. From ~30–58 h it sits near ~2 km (flat/plateau). Around ~60 h it jumps up to ~55–60 km, then rapidly contracts: ~25 km by ~63 h, ~15–20 km by ~70 h, ~10–12 km by ~80–90 h. From ~80–120 h it is mostly ~10–14 km with small step-like changes; after ~150 h it increases gradually to ~14–18 km by ~170–195 h, ending near ~18 km.

Bottom-right (Max vert. velocity): y-axis “Max vert. velocity (m/s)”. Near 0 m/s at 0 h; rises quickly after ~5–10 h with early sharp spikes up to ~13–15 m/s around ~20–30 h. After ~30 h it becomes noisy, generally 2–9 m/s with intermittent spikes >10 m/s (including a peak near ~15 m/s around ~95–100 h). Late period (~150–200 h) stays mostly ~6–10 m/s, ending ~7–8 m/s.

**X-axis:** Time (h), 0–200

**Y-axis:** Multiple panels; y-axes are: Max horiz. wind (m/s) ~10–95; Min sfc pressure (hPa) ~930–1012; RMW (km) ~0–90; Max vert. velocity (m/s) ~0–15

**Notes:** RMW time series contains repeated near-zero values and near-vertical jumps (≈18–30 h) and a long near-constant ~2 km segment (≈30–58 h), suggestive of tracking/diagnostic discontinuities rather than smooth physical evolution.

**Legend:**
- output — blue solid

### 2. `gw9fnuo` — _plot_

![gw9fnuo](https://s.odsi.io/gw9fnuo)

**Caption:** RQ3 H3.1 — 10 m wind and convective depth

**Description:** Composite figure (2-panel line_plot time series) titled “RQ3 H3.1 — 10 m wind and convective depth”. Single blue line labeled “output” in each panel.

Left (Max 10 m wind): y-axis “Max 10 m wind (m/s)”. Starts ~13–14 m/s at 0 h, dips to ~10 m/s by ~5–10 h, then rises to ~15–20 m/s by ~30–45 h. Stays around ~18–24 m/s through ~60 h, followed by rapid increase ~65–85 h from ~20–30 m/s to ~50–60 m/s. Post ~85 h remains high and variable, mostly ~55–66 m/s, with dips near ~100 h (~50–55 m/s) and ~155 h (~50 m/s). Late period (~165–190 h) reaches ~66–68 m/s before ending near ~62–63 m/s.

Right (Cloud top height): y-axis “Cloud top height (km)”. Near 0 km at start, rises rapidly to ~10–12 km by ~10 h and ~15 km by ~15–18 h. Has an early spike near ~21 km around ~22–25 h. From ~25–150 h it fluctuates mostly ~15–18 km with occasional peaks near ~19–20 km. After ~150 h it gradually trends slightly lower, hovering ~15–16 km, ending near ~15–15.5 km.

**X-axis:** Time (h), 0–200

**Y-axis:** Left: Max 10 m wind (m/s) ~8–70; Right: Cloud top height (km) ~0–21

**Legend:**
- output — blue solid

### 3. `8lnuoxm` — _plot_

![8lnuoxm](https://s.odsi.io/8lnuoxm)

**Caption:** RQ3 H3.1 — Theta-e and surface vorticity

**Description:** Composite figure (2-panel line_plot time series) titled “RQ3 H3.1 — Theta-e and surface vorticity”. Single blue line labeled “output” in each panel.

Left (Max theta-e below 10 km): y-axis “Max theta-e below 10 km (K)”. Begins near ~361 K at 0 h, rapidly jumps to ~367–368 K by ~5–10 h. Slight decline/plateau ~367–368 K through ~30 h, then drops to ~366–366.5 K around ~30–35 h and stays relatively steady near ~366.5 K until ~75 h. After ~75–80 h becomes much more variable, oscillating roughly ~366–371 K with intermittent spikes above ~372 K (notably around ~115 h and ~175–185 h). Ends near ~369.5–370 K.

Right (Max sfc vorticity): y-axis “Max sfc vorticity (1/s)”. Starts very small (~0.0003–0.0005 1/s) and increases gradually to ~0.002 by ~55–60 h. Rapid ramp-up occurs ~60–80 h to ~0.012–0.017. Peaks around ~0.020–0.021 near ~85–100 h. After ~100 h remains elevated but noisy, mostly ~0.013–0.018, with occasional sharp dips (e.g., around ~150 h to below ~0.010). Declines slightly toward the end, finishing near ~0.012–0.013.

**X-axis:** Time (h), 0–200

**Y-axis:** Left: Max theta-e below 10 km (K) ~361–373; Right: Max sfc vorticity (1/s) ~0.000–0.021

**Legend:**
- output — blue solid

### 4. `ktanj05` — _plot_

![ktanj05](https://s.odsi.io/ktanj05)

**Caption:** RQ3 H3.1 — BL wind and PBL depth vs stability

**Description:** Composite figure (2-panel line_plot time series) titled “RQ3 H3.1 — BL wind and PBL depth vs stability”. Single blue line labeled “output” in each panel.

Left (Max horiz. wind at lowest level): y-axis “Max horiz. wind at lowest level (m/s)”. Starts ~14 m/s at 0 h, dips to ~10–11 m/s by ~5–10 h, then rises to ~20–25 m/s by ~40–55 h. Rapid increase ~60–85 h from ~20–30 m/s to ~55–65 m/s. After ~85 h remains high with fluctuations mostly ~60–75 m/s; local peaks around ~75–76 m/s near ~120–130 h and again ~175–185 h, with noticeable dips near ~100–110 h and ~150–155 h. Ends near ~69–70 m/s.

Right (Max PBL depth): y-axis “Max PBL depth (km)”. Starts near ~0.45–0.5 km at 0 h, rises quickly to ~0.7–0.8 km by ~5–10 h, then slowly increases to ~0.9–1.0 km by ~45–55 h. Sharp deepening occurs ~60–90 h, increasing to ~1.4–1.9 km by ~75–85 h and ~2.0–2.3 km by ~95–110 h. Thereafter fluctuates around ~2.0–2.5 km with spikes up to ~2.7 km (around ~125 h) and a larger spike near ~3.1 km around ~168–170 h. Ends near ~2.5–2.6 km.

**X-axis:** Time (h), 0–200

**Y-axis:** Left: Max horiz. wind at lowest level (m/s) ~8–78; Right: Max PBL depth (km) ~0.4–3.2

**Legend:**
- output — blue solid

### 5. `44n6qvs` — _plot_

![44n6qvs](https://s.odsi.io/44n6qvs)

**Caption:** RQ3 H3.1 — Lowest-model-level wind components vs stability

**Description:** Composite figure (4-panel line_plot time series) titled “RQ3 H3.1 — Lowest-model-level wind components vs stability”. Single blue line labeled “output” in each panel.

Top-left (Max u at lowest level): y-axis “Max u at lowest level (m/s)”. Near 0 m/s at 0–~5 h, then becomes highly intermittent: generally ~1–7 m/s with frequent spikes. Notable peaks include ~11 m/s around ~30 h and ~12–13 m/s around ~60–70 h. After ~120 h remains variable; around ~135–145 h values drop near ~0.5–1 m/s, then rebound with spikes (~8–10 m/s) around ~150–165 h. Ends near ~2–3 m/s.

Top-right (Min u at lowest level): y-axis “Min u at lowest level (m/s)”. Starts near 0 m/s and steadily becomes more negative to ~-5 m/s by ~15–25 h and ~-8 to -10 m/s by ~50–70 h. Sharp transition ~70–90 h to much stronger negatives, reaching ~-25 to -35 m/s (deepest values near ~-37 to -39 around ~95–105 h and again late in the run). After ~90 h remains very negative with large variability (roughly ~-20 to -38 m/s), ending near ~-37 m/s.

Bottom-left (Max v at lowest level): y-axis “Max v at lowest level (m/s)”. Starts ~13–14 m/s, dips toward ~10–11 m/s by ~5–10 h, then rises to ~20–25 m/s by ~40–55 h. Rapid increase ~60–85 h up to ~55–65 m/s, then fluctuates mostly ~60–73 m/s thereafter with a local maximum ~73–74 m/s around ~120–130 h. Ends near ~66–67 m/s.

Bottom-right (Min v at lowest level): y-axis “Min v at lowest level (m/s)”. Mostly near 0 m/s (slightly negative) for long periods, but punctuated by sharp negative spikes. A very large spike reaches about ~-27 m/s near ~70–75 h, followed by recovery to near 0. Another period of frequent spikes occurs ~135–180 h with minima reaching roughly ~-15 to -22 m/s. Ends close to 0 to ~-1 m/s.

**X-axis:** Time (h), 0–200

**Y-axis:** Multiple panels; y-axes are: Max u (m/s) ~0–14; Min u (m/s) ~0 to -40; Max v (m/s) ~10–75; Min v (m/s) ~0 to -28

**Notes:** The “Min v at lowest level” series shows isolated extreme negative spikes (e.g., ~-27 m/s near ~70–75 h) with near-zero values otherwise, suggesting possible sampling/tracking discontinuities or intermittent sign/sector selection effects.

**Legend:**
- output — blue solid

### 6. `2e6v14x` — _plot_

![2e6v14x](https://s.odsi.io/2e6v14x)

**Caption:** RQ3 H3.1 — Vertical vorticity at multiple levels vs stability

**Description:** Composite figure (6-panel line_plot time series) titled “RQ3 H3.1 — Vertical vorticity at multiple levels vs stability”. Each panel shows a single blue line labeled “output”, with x-axis Time (h).

Top-left (surface): y-axis “Max vorticity sfc (1/s)”. Gradual increase from ~0.0003–0.0005 to ~0.002 by ~55–60 h, then sharp rise ~60–80 h to ~0.015–0.018 and peaks near ~0.020 around ~85–100 h. Afterward fluctuates ~0.013–0.018 with occasional dips (down to ~0.009–0.011), ending near ~0.012–0.013.

Top-middle (1 km): y-axis “Max vorticity 1 km (1/s)”. Starts near ~0.0003 and rises to ~0.002–0.003 by ~50–60 h. Rapid increase ~60–75 h to ~0.013–0.016. Contains multiple sharp peaks above ~0.020 around ~95–115 h. Post-peak fluctuates roughly ~0.010–0.016 and ends near ~0.012–0.013.

Top-right (2 km): y-axis “Max vorticity 2 km (1/s)”. Increases slowly to ~0.003 by ~55–60 h, then jumps ~60–75 h to ~0.012–0.013. Peaks near ~0.018–0.019 around ~85–95 h and then settles ~0.010–0.013 with a mild downward trend, ending near ~0.010.

Bottom-left (3 km): y-axis “Max vorticity 3 km (1/s)”. Early values near zero with an isolated spike near ~0.004–0.005 around ~20–25 h. Gradually rises to ~0.002–0.003 by ~55–60 h, then increases rapidly ~60–80 h to ~0.010–0.013 with an early peak around ~0.017 near ~80 h. After ~90 h declines and fluctuates mostly ~0.008–0.011, reaching ~0.007–0.009 by ~150–200 h.

Bottom-middle (4 km): y-axis “Max vorticity 4 km (1/s)”. Similar pattern: near-zero early with a spike around ~20–25 h (~0.003–0.004), gradual rise to ~0.002–0.003 by ~60 h, then sharp increase ~60–85 h to ~0.009–0.012 and peaks around ~0.015–0.016 near ~90–100 h. Declines toward ~0.006–0.008 by ~150–200 h.

Bottom-right (5 km): y-axis “Max vorticity 5 km (1/s)”. Near-zero early with small spike ~20–25 h (~0.002–0.003), gradual rise to ~0.002 by ~55–60 h, then jump ~60–85 h to ~0.009–0.012 and peak near ~0.016 around ~90–95 h. Later decays to ~0.006–0.007 by ~150–200 h.

**X-axis:** Time (h), 0–200

**Y-axis:** Multiple panels; y-axes are max vertical vorticity (1/s) at sfc, 1, 2, 3, 4, 5 km (ranges approx: 0–0.022 depending on level)

**Legend:**
- output — blue solid

### 7. `4l5iu2m` — _plot_

![4l5iu2m](https://s.odsi.io/4l5iu2m)

**Caption:** RQ3 H3.1 — Pressure diagnostics vs stability

**Description:** Composite figure (3-panel line_plot time series) titled “RQ3 H3.1 — Pressure diagnostics vs stability”. Single blue line labeled “output” in each panel.

Left (Min sfc pressure): y-axis “Min sfc pressure (hPa)”. Starts ~1009–1010 hPa, decreases gradually to ~996–997 by ~65–70 h, then drops sharply ~70–85 h to ~945 hPa. Thereafter oscillates strongly mostly ~935–955 hPa with local minima near ~930 hPa (notably ~125 h and ~185–190 h). Ends ~936–938 hPa.

Middle (Max sfc pressure): y-axis “Max sfc pressure (hPa)”. Very narrow range around ~1014.8–1015.4 hPa. Begins near ~1014.80–1014.82, rises toward ~1014.95–1015.05 by ~20–40 h. Contains intermittent spikes up to ~1015.35–1015.45 around ~50–105 h. After ~110 h fluctuates around ~1014.90–1015.00 with occasional dips to ~1014.80–1014.85 (notably ~165–175 h). Ends near ~1014.88–1014.90.

Right (Pressure deficit): y-axis “Pressure deficit (hPa)”. Starts around ~6 hPa, increases steadily to ~10–15 by ~40–60 h and ~18–20 by ~70 h. Rapid jump ~75–90 h up to ~70–80 hPa. Thereafter remains elevated with oscillations roughly ~60–85 hPa; ends around ~78–82 hPa.

**X-axis:** Time (h), 0–200

**Y-axis:** Left: Min sfc pressure (hPa) ~930–1012; Middle: Max sfc pressure (hPa) ~1014.8–1015.45; Right: Pressure deficit (hPa) ~5–90

**Notes:** The “Max sfc pressure” panel has an extremely compressed y-range (~0.6 hPa total), making small fluctuations appear large; interpretation should account for this scaling.

**Legend:**
- output — blue solid

### 8. `1ke60l5` — _plot_

![1ke60l5](https://s.odsi.io/1ke60l5)

**Caption:** RQ3 H3.1 — Convective metrics vs stability

**Description:** Composite figure (6-panel line_plot time series) titled “RQ3 H3.1 — Convective metrics vs stability”. Single blue line labeled “output” in each panel.

Top-left (Max w): y-axis “Max w (m/s)”. Near 0 at start, rises quickly after ~5–10 h with early spikes up to ~13–15 m/s around ~20–30 h. Thereafter noisy, typically ~2–10 m/s with intermittent spikes >10 m/s through the run; late times (~150–200 h) often ~6–9 m/s.

Top-middle (Min w): y-axis “Min w (m/s)”. Starts near 0 and becomes negative within ~5–10 h. Early period shows strong negative spikes (downdrafts) down to about ~-8 to -9 m/s around ~15–25 h. After ~30 h generally stays around ~-1 to -3 m/s with sporadic deeper dips (~-4 to -6 m/s) around ~80–110 h and occasional later dips.

Top-right (Height of max w): y-axis “Height of max w (km)”. Near 0 at start; increases rapidly to several km by ~10–15 h. Between ~15–70 h it is highly variable, with frequent spikes reaching ~15–22 km and drops to a few km. After ~80 h it stabilizes mostly around ~4–7 km with intermittent brief excursions (down to ~1–3 km and up to ~8–10 km), ending near ~6–7 km.

Bottom-left (Cloud top): y-axis “Cloud top (km)”. Near 0 initially, rises to ~10–12 km by ~10 h and ~15–16 km by ~15–20 h, with an early peak near ~20–21 km around ~20–25 h. Then fluctuates mostly ~15–18 km, with a slight downward tendency after ~150 h to ~15–16 km by the end.

Bottom-middle (Cloud base): y-axis “Cloud base (km)”. Starts at 0, spikes briefly to ~0.55–0.6 km very early, then mostly remains in the ~0.17–0.30 km range with step-like plateaus. Around ~110–140 h there is a sustained lower period near ~0.09–0.17 km, with occasional spikes back toward ~0.25–0.30 km. Ends near ~0.17 km with intermittent dips.

Bottom-right (Cloud depth): y-axis “Cloud depth (km)”. Rapidly increases from ~0 to ~14–16 km by ~15–25 h, then remains fairly steady around ~15–17 km with modest fluctuations; slight reduction toward ~15–16 km late in the run.

**X-axis:** Time (h), 0–200

**Y-axis:** Multiple panels; y-axes are: Max w (m/s) ~0–15; Min w (m/s) ~0 to -9; Height of max w (km) ~0–22; Cloud top (km) ~0–21; Cloud base (km) ~0–0.6; Cloud depth (km) ~0–21

**Notes:** Cloud base exhibits quantized/step-like behavior (extended flat segments at discrete values), consistent with discretization or threshold-based diagnostic output rather than a smoothly varying quantity.

**Legend:**
- output — blue solid

### 9. `ksi6x4l` — _plot_

![ksi6x4l](https://s.odsi.io/ksi6x4l)

**Caption:** RQ3 H3.1 — Theta-e and moisture vs stability

**Description:** Composite figure (4-panel line_plot time series) titled “RQ3 H3.1 — Theta-e and moisture vs stability”. Single blue line labeled “output” in each panel.

Top-left (Max theta-e < 10 km): y-axis “Max theta-e < 10 km (K)”. Starts ~361 K and jumps to ~367–368 K by ~5–10 h. Decreases toward ~366–366.5 K around ~30–35 h and remains near ~366.5 K until ~75 h. After ~75 h becomes highly variable with oscillations ~366–371+ K and spikes above ~372 K (notably around ~115 h and ~175–185 h). Ends near ~369.5–370 K.

Top-right (Max theta-e at lowest level): y-axis “Max theta-e at lowest level (K)”. Very similar early evolution: ~361 K at 0 h, quick rise to ~367–368 K by ~5–10 h, then drop to ~366–366.5 K around ~30–35 h and steady until ~75 h. After ~75 h shows strong variability with multiple peaks up to ~372–373 K; late peaks around ~175–185 h, ending near ~369–370 K.

Bottom-left (Max qv): y-axis “Max qv (g/kg)”. Begins near ~20.3 g/kg and increases sharply to ~22.5–22.7 g/kg within the first few hours. Then stays in a narrow band (~22.3–22.9 g/kg) with gentle oscillations; slight upward drift later with values approaching ~22.9–23.0 g/kg around ~150–190 h. Ends near ~22.7–22.8 g/kg.

Bottom-right (Min theta-e at lowest level): y-axis “Min theta-e at lowest level (K)”. Starts around ~353.6 K and slowly rises toward ~354–355 K by ~40–60 h. From ~50–110 h becomes jagged with intermittent sharp drops; a pronounced minimum reaches approximately ~346 K near ~85–90 h. After ~120 h there is a longer downward period to ~351 K by ~155–165 h with multiple sharp dips (including ~347–349 K near ~125–130 h). Around ~170–175 h it rebounds sharply to ~355–356 K, then declines again to ~353 K by ~195–200 h.

**X-axis:** Time (h), 0–200

**Y-axis:** Multiple panels; y-axes are: Max theta-e <10 km (K) ~361–373; Max theta-e lowest level (K) ~361–373; Max qv (g/kg) ~20.3–23.0; Min theta-e lowest level (K) ~346–356

**Notes:** The “Min theta-e at lowest level” series contains abrupt deep drops (down to ~346 K) and rapid recoveries, indicating strong intermittency or possible sensitivity to how the minimum is sampled (e.g., localized outliers dominating the minimum).

**Legend:**
- output — blue solid

### 10. `lth7kni` — _plot_

![lth7kni](https://s.odsi.io/lth7kni)

**Caption:** RQ3 H3.1 — Precipitation and mass flux vs stability

**Description:** Composite figure with three time-series line plots (subtype: line_plot), all showing a single series labeled “output” (blue solid line) versus time.

Left panel: “Max precip rate (kg/m^2/s)” vs Time.
- Starts near 0 at 0 h and remains ~0–0.005 through ~10 h.
- Early sharp spike to ~0.065 around ~20–25 h (largest value in the panel), immediately returning to ~0.01–0.02.
- From ~30 to ~70 h: noisy, generally increasing envelope, mostly ~0.005–0.02 with intermittent spikes up to ~0.03–0.035.
- From ~70 to ~130 h: higher plateau-like regime, fluctuating mostly ~0.02–0.04 with peaks ~0.045–0.046 around ~110–130 h.
- After ~130 h: decreases to ~0.02–0.03 with continued noise; ends around ~0.022–0.024 by ~190 h.

Middle panel: “Total upward mass flux (kg m/s)” vs Time (axis shows scientific scale marker “1e12”, so values are O(10^12)).
- Begins near 0 at 0 h; rises slowly to ~0.2×10^12 by ~10 h.
- Rapid increase to ~1.3–1.5×10^12 by ~20–30 h.
- Continues increasing with strong variability, reaching ~2.0–2.4×10^12 by ~50–80 h.
- Peaks near ~2.7–2.8×10^12 around ~95–105 h.
- Sharp drop to ~1.1–1.3×10^12 around ~110 h.
- 110–170 h: oscillatory with multiple peaks/troughs; notable secondary peak ~2.6×10^12 around ~150–155 h and a dip near ~0.9–1.0×10^12 around ~140 h.
- Late period: overall decline, ending near ~1.0×10^12 by ~190 h.

Right panel: “Total downward mass flux (kg m/s)” vs Time (also with “1e12”; values are negative).
- Starts near 0 at 0 h and becomes more negative quickly: ~-1.0×10^12 by ~20 h.
- 20–60 h: fluctuates mostly between ~-1.3 and ~-2.3×10^12, trending more negative.
- Deepest minimum near ~-3.4×10^12 around ~100 h (most negative point).
- After ~100 h: rebounds (less negative) to around ~-1.2 to ~-1.5×10^12 by ~115–125 h.
- 125–160 h: renewed variability with dips near ~-2.7 to ~-2.9×10^12 around ~150 h and intermittent recovery to ~-1.1×10^12.
- Late period: gradual recovery toward ~-1.1×10^12 by ~190 h.

**X-axis:** Time (h), 0–200 (all three panels)

**Y-axis:** Left: Max precip rate (kg/m^2/s), ~0–0.065; Middle: Total upward mass flux (kg m/s), ~0–2.8×10^12; Right: Total downward mass flux (kg m/s), ~0 to -3.5×10^12

**Legend:**
- output — blue solid

### 11. `lfpoeb8` — _plot_

![lfpoeb8](https://s.odsi.io/lfpoeb8)

**Caption:** RQ3 H3.1 — RMW vs max wind (scatter)

**Description:** Single-panel scatter plot (subtype: scatter_plot) titled “RQ3 H3.1 — RMW vs max wind (scatter)”. Points are semi-transparent blue circles labeled “output”.

Axes and overall pattern:
- X: RMW (km), spanning roughly 0 to ~90 km.
- Y: Max wind (m/s), spanning roughly ~10 to ~95 m/s.
- The point cloud is not a single linear trend; instead it forms several clusters, with strongest winds concentrated at intermediate RMW (~10–20 km), and generally weaker winds at very large RMW (>~40–50 km). Very small RMW (~0–2 km) also corresponds mainly to low winds.

Notable clusters/outliers (approximate):
- Dense vertical cluster at very small RMW (~0–2 km): winds mostly ~14–33 m/s (many points between ~15–25 m/s).
- Dense cluster around RMW ~10–12 km: winds primarily high, ~63–88 m/s (many points ~70–85 m/s).
- Cluster around RMW ~14–16 km: winds ~72–89 m/s, with a few mid/low-wind points down near ~40–65 m/s.
- Cluster around RMW ~18–19 km: many points ~80–92 m/s, but also several lower points ~34–40 m/s.
- Sparse points at larger RMW (~20–90 km): mostly low winds ~12–26 m/s; a few mid-wind points around ~30–35 m/s near ~20–25 km.
- Highest winds are near ~90–92 m/s at RMW ~18–19 km (top of the high-wind cluster).

**X-axis:** RMW (km), ~0–90

**Y-axis:** Max wind (m/s), ~10–95

**Legend:**
- output — blue circle markers

### 12. `c65grax` — _plot_

![c65grax](https://s.odsi.io/c65grax)

**Caption:** RQ3 H3.1 — Intensity change rate vs stability

**Description:** Composite figure with two time-series line plots (subtype: line_plot), each showing one blue series labeled “output” and a dashed horizontal zero reference line.

Left panel: dVmax/dt (m/s per hr) vs Time (h).
- Values hover around 0 most of the time, with frequent high-frequency fluctuations typically within about ±5 m/s/hr.
- Early period (0–20 h): near 0 with small oscillations, then a positive spike around ~20–25 h reaching ~+7 to +8.
- Mid period (30–90 h): repeated spikes and dips; several positive excursions up to ~+10–+12 (e.g., around ~50 h and ~75–80 h), and negative dips to ~-5 to ~-7.
- Around ~95–105 h: the most extreme variability: a sharp positive spike up to ~+23 (near ~98–100 h) followed closely by a deep negative dip down to ~-14 (near ~100 h).
- 110–160 h: oscillatory with multiple negative dips around ~-10 to ~-12 (notably near ~120–130 h and ~155 h) and intermittent positive spikes to ~+10–+12.
- Late period (160–195 h): variability decreases somewhat; values mostly within ~±3, ending near ~0 to slightly negative.

Right panel: dPmin/dt (hPa/hr) vs Time (h).
- Early period (0–20 h): near 0 with small oscillations.
- 20–70 h: frequent fluctuations around zero, many within about ±3–4 hPa/hr.
- 70–105 h: strongest deepening episodes (negative): several sharp negative spikes, including dips near ~-10 to ~-11 (around ~80–85 h) and the deepest dip near ~-14 (around ~95–100 h).
- 105–160 h: continued large-amplitude oscillations; includes positive spikes up to ~+8 (around ~125 h) and multiple negative spikes down to ~-6 to ~-9.
- 160–195 h: oscillations narrow toward ~-3 to +3; ends near ~0 to slightly positive.

Overall, both rate series are highly non-monotonic and noisy, with the largest-magnitude rate events clustered around ~75–110 h.

**X-axis:** Time (h), 0–200 (both panels)

**Y-axis:** Left: dVmax/dt (m/s per hr), ~-15–25; Right: dPmin/dt (hPa/hr), ~-15–9

**Legend:**
- output — blue solid

### 13. `68aiwqs` — _plot_

![68aiwqs](https://s.odsi.io/68aiwqs)

**Caption:** RQ3 H3.1 — Summary bar charts (peak metrics)

**Description:** Composite figure of four single-category bar charts (subtype: bar_chart), each with one blue bar labeled “output” on the x-axis, summarizing peak metrics.

Panels (left to right):
1) “Peak Vmax (m/s)”: single bar near the low 90s m/s (approximately ~92 m/s).
2) “Min Psfc (hPa)”: single bar near the low-to-mid 900s hPa (approximately ~930–940 hPa; bar top appears slightly above 900 with y-ticks shown at 0,200,400,600,800).
3) “Peak Wmax (m/s)”: single bar near ~15 m/s (approximately ~15).
4) “Min RMW (km)”: single bar at ~2.0 km (approximately 2.0).

No error bars or significance annotations are shown; each panel contains only the single “output” category.

**X-axis:** Category axis with single label “output” (all panels)

**Y-axis:** Panel 1: Peak Vmax (m/s), ~0–95; Panel 2: Min Psfc (hPa), ~0–950; Panel 3: Peak Wmax (m/s), ~0–15+; Panel 4: Min RMW (km), ~0–2.0

**Notes:** The “Min Psfc (hPa)” bar chart uses a zero-based y-axis (0 to ~900+ hPa), which compresses variability and is atypical for pressure visualization; the plotted value is near ~930–940 hPa but hard to read precisely due to scaling.

### 14. `c4whnhs` — _plot_

![c4whnhs](https://s.odsi.io/c4whnhs)

**Caption:** RQ3 H3.1 — Intensity and structure vs environmental stability

**Description:** Composite 2×2 figure of time-series line plots (subtype: line_plot) titled “RQ3 H3.1 — Intensity and structure vs environmental stability”. Each panel shows a single blue “output” line versus Time (h).

Top-left: “Max horiz. wind (m/s)” vs Time.
- Starts ~14–15 m/s at 0 h; rises to ~25 m/s by ~10 h.
- 10–30 h: oscillates ~20–27 m/s.
- Rapid intensification 30–55 h: climbs from ~25–30 to ~70–75 m/s.
- 55–100 h: continues upward to ~85–95 m/s with fluctuations.
- 105–120 h: noticeable weakening, dropping to ~70–80 m/s (minimum in this mature phase around ~72).
- 125–190 h: re-intensifies and remains high; frequently ~95–108 m/s, ending near ~100–103 m/s.

Top-right: “Min sfc pressure (hPa)” vs Time.
- Starts near ~1010 hPa.
- Gradual fall to ~995–1000 by ~20 h, then ~980 by ~40 h.
- Sharp drop around ~50–60 h to ~950 hPa.
- Further decline to ~930–935 by ~75–85 h.
- 85–130 h: fluctuates with partial rebounds; roughly ~925–940 hPa.
- 130–190 h: trends lower; around ~915–920 by ~150–160 h and reaches a minimum near ~900 hPa around ~180–190 h; ends ~905 hPa.

Bottom-left: “RMW (km)” vs Time.
- Begins large (~75–80 km), then decreases to ~30–40 km by ~10–15 h.
- Very early period (~15–25 h) shows extreme spikes and rapid jumps, including peaks above ~200 km (largest ~210–220 km), interspersed with drops to ~15–30 km.
- 25–55 h: generally decreases from ~40–60 km toward ~15–20 km.
- 55–100 h: stays low and relatively steady ~15–18 km.
- Stepwise increases later: around ~105 h rises to ~20–22 km; around ~125–135 h rises to ~28–30 km.
- 140–190 h: mostly ~25–27 km with minor fluctuations.

Bottom-right: “Max vert. velocity (m/s)” vs Time.
- Near 0 at 0 h, then rapid jump to ~10 m/s by ~5 h.
- 5–20 h: strong early variability with peaks ~12–13 m/s.
- 20–45 h: weaker interval ~3–6 m/s.
- 45–110 h: increases and fluctuates broadly ~5–10 m/s; one prominent peak near ~19 m/s around ~85–90 h.
- 110–125 h: dips to ~4–7 m/s.
- 130–190 h: renewed variability; often ~6–12 m/s, ending around ~9–11 m/s.

Across panels, intensification (rising winds, falling pressure) occurs mainly between ~30 and ~80 h, with later oscillations and a midlife weakening (~110–120 h) followed by re-intensification.

**X-axis:** Time (h), 0–200 (all panels)

**Y-axis:** Top-left: Max horiz. wind (m/s), ~10–110; Top-right: Min sfc pressure (hPa), ~900–1015; Bottom-left: RMW (km), ~10–225; Bottom-right: Max vert. velocity (m/s), ~0–20

**Notes:** RMW time series shows very large, short-lived spikes early (~15–25 h, up to >200 km), which may indicate a tracking/diagnostic instability or spin-up artifact rather than smooth structural evolution.

**Legend:**
- output — blue solid

### 15. `3ar1r65` — _plot_

![3ar1r65](https://s.odsi.io/3ar1r65)

**Caption:** RQ3 H3.1 — 10 m wind and convective depth

**Description:** Composite figure with two time-series line plots (subtype: line_plot) titled “RQ3 H3.1 — 10 m wind and convective depth”. Both panels show one blue “output” line vs Time (h).

Left: “Max 10 m wind (m/s)” vs Time.
- Starts ~11–14 m/s at 0 h.
- Increases to ~18–21 m/s by ~10–20 h, with small oscillations.
- 25–55 h: strong intensification from ~20 to ~55–57 m/s.
- 55–105 h: slower increase/plateau around ~55–68 m/s; reaches ~65–69 m/s around ~80–100 h.
- 110–125 h: temporary weakening to ~54–58 m/s.
- 125–190 h: recovery and gradual increase, frequently ~66–76 m/s; ends ~72–74 m/s.

Right: “Cloud top height (km)” vs Time.
- Near 0 at 0 h, then extremely rapid deepening: rises to ~18–19 km by ~5–10 h.
- 10–50 h: remains very deep with frequent high-frequency fluctuations, typically ~16–19 km.
- 50–90 h: slightly lower mean, mostly ~15–17.5 km.
- 90–190 h: becomes more stable/flat near ~15.5–16.5 km, with occasional brief spikes up to ~18 km; ends near ~16 km.

Overall, cloud tops reach deep convection very early and then remain near a quasi-steady deep-convective height while near-surface wind intensifies more gradually.

**X-axis:** Time (h), 0–200 (both panels)

**Y-axis:** Left: Max 10 m wind (m/s), ~10–80; Right: Cloud top height (km), ~0–20

**Legend:**
- output — blue solid

### 16. `jsht0kt` — _plot_

![jsht0kt](https://s.odsi.io/jsht0kt)

**Caption:** RQ3 H3.1 — Theta-e and surface vorticity

**Description:** Composite figure with two time-series line plots (subtype: line_plot) titled “RQ3 H3.1 — Theta-e and surface vorticity”. Both panels show one blue “output” line vs Time (h).

Left: “Max theta-e below 10 km (K)” vs Time.
- Starts around ~360–361 K at 0 h.
- Brief early rise to ~365 K by ~5 h, followed by a drop back to ~360–361 K by ~10–15 h.
- 15–45 h: relatively flat, mostly ~359–362 K, with small noise.
- 45–105 h: gradual increase; reaches ~365–368 K by ~70–100 h, with intermittent small spikes.
- 110–160 h: stronger upward trend; rises from ~369–371 K to ~380–382 K by ~150–165 h.
- 170–190 h: small dip near ~378–379 K around ~175 h, then resumes rising, ending near ~384 K.

Right: “Max sfc vorticity (1/s)” vs Time.
- Begins very low (~0.0005–0.001 1/s).
- Increases steadily to ~0.003 by ~25 h and ~0.005–0.006 by ~35–45 h.
- Rapid intensification around ~45–60 h: jumps to ~0.011 by ~50 h and peaks near ~0.018–0.019 around ~55–65 h.
- 65–105 h: fluctuates at elevated values ~0.012–0.016.
- 105–130 h: decreases to a lower regime ~0.008–0.011 (minimum in this later period near ~0.007–0.008 around ~120–125 h).
- 130–190 h: gradual recovery and fluctuations mostly ~0.010–0.013; ends near ~0.012.

The two series both show overall increases over the simulation, with vorticity peaking earlier (~60 h) and theta-e continuing to rise later into the run.

**X-axis:** Time (h), 0–200 (both panels)

**Y-axis:** Left: Max theta-e below 10 km (K), ~358–385; Right: Max sfc vorticity (1/s), ~0.000–0.019

**Legend:**
- output — blue solid

### 17. `a88bl6i` — _plot_

![a88bl6i](https://s.odsi.io/a88bl6i)

**Caption:** RQ3 H3.1 — BL wind and PBL depth vs stability

**Description:** Composite figure with two time-series line plots (subtype: line_plot) titled “RQ3 H3.1 — BL wind and PBL depth vs stability”. Both panels show one blue “output” line vs Time (h).

Left: “Max horiz. wind at lowest level (m/s)” vs Time.
- Starts ~11–15 m/s at 0 h.
- 0–25 h: increases gradually to ~18–22 m/s.
- 25–55 h: rapid intensification from ~20 to ~60+ m/s (around ~60–63 by ~55 h).
- 55–100 h: continues rising into ~70–75 m/s with oscillations.
- 105–125 h: noticeable weakening to ~60–66 m/s.
- 125–190 h: recovery and continued strengthening; commonly ~75–86 m/s after ~140 h; ends near ~80–83 m/s.

Right: “Max PBL depth (km)” vs Time.
- Starts around ~0.7–0.8 km at 0 h.
- Increases to ~1.1–1.2 km by ~5–10 h; remains near ~1.1–1.3 km through ~25–30 h.
- 30–55 h: gradual growth to ~1.7–1.9 km.
- Around ~55–60 h: step-like increase to ~2.2–2.4 km.
- 60–120 h: fluctuates ~2.2–2.7 km, with many short-period oscillations.
- ~125–140 h: increases into ~2.8–3.1 km.
- 140–190 h: remains high, typically ~2.8–3.2 km, with a peak near ~3.3 km late in the run; ends around ~3.0 km.

Overall, lowest-level wind and PBL depth both increase over time with a pronounced transition around ~50–60 h, after which PBL depth stays in a deeper regime (~2–3+ km).

**X-axis:** Time (h), 0–200 (both panels)

**Y-axis:** Left: Max horiz. wind at lowest level (m/s), ~10–90; Right: Max PBL depth (km), ~0.7–3.3

**Legend:**
- output — blue solid

### 18. `ddmnhmh` — _plot_

![ddmnhmh](https://s.odsi.io/ddmnhmh)

**Caption:** RQ3 H3.1 — Lowest-model-level wind components vs stability

**Description:** Composite figure (4-panel line plots) titled “RQ3 H3.1 — Lowest-model-level wind components vs stability”. All panels show a single time series labeled “output”. X-axis is time in hours.

Top-left: line_plot of “Max u at lowest level (m/s)” vs time. Starts near 0 m/s at 0 h, rises sharply to ~11–12 m/s by ~10–20 h. After ~20 h, fluctuates strongly (noisy, non-monotonic) mostly between ~3 and ~9 m/s. Notable dips to ~2–3 m/s around ~40–50 h and again near ~150–160 h (briefly near ~0–1 m/s). Occasional spikes back to ~10–12 m/s around ~110–120 h. Ends around ~6–7 m/s near ~190–195 h.

Top-right: line_plot of “Min u at lowest level (m/s)” vs time. Begins near 0 m/s, quickly becomes negative within the first ~5–10 h (around −5 to −10 m/s). Overall trend becomes more negative with time: around −10 to −15 m/s by ~20–40 h; around −20 to −35 m/s by ~60–110 h with substantial variability; then a sharper drop near ~125 h to around −40 m/s. After ~130 h remains mostly between ~−35 and −45 m/s, with a deepest excursion near ~180–185 h approaching ~−50 m/s. Ends near ~−42 to −45 m/s.

Bottom-left: line_plot of “Max v at lowest level (m/s)” vs time. Starts ~12–15 m/s at 0 h, rises gradually with a marked intensification phase from ~30–60 h (climbing from ~20–30 m/s to ~55–60 m/s). Continues to increase to ~65–70 m/s by ~70–100 h, then has a dip/plateau around ~105–125 h (~58–65 m/s). After ~125 h rises again, reaching ~75–82 m/s by ~140–170 h. Near the end (~170–195 h) stays high around ~76–80 m/s.

Bottom-right: line_plot of “Min v at lowest level (m/s)” vs time. Values are near 0 m/s for much of 0–30 h. Between ~30–80 h, repeated sharp negative spikes occur, ranging roughly from ~−3 to ~−15 m/s, with one deepest spike near ~55 h reaching about −22 m/s. After ~85–120 h, values hover close to 0 to slightly negative (~0 to ~−1 m/s). After ~125 h, becomes mildly negative (~−0.5 to −2 m/s) with small fluctuations; ends around ~−1 m/s.

**X-axis:** Time (h), 0–200

**Y-axis:** Varies by panel: Max u at lowest level (m/s) ~0–12; Min u at lowest level (m/s) ~0 to −50; Max v at lowest level (m/s) ~10–85; Min v at lowest level (m/s) ~0 to −22

**Legend:**
- output — blue solid

### 19. `ns84skq` — _plot_

![ns84skq](https://s.odsi.io/ns84skq)

**Caption:** RQ3 H3.1 — Vertical vorticity at multiple levels vs stability

**Description:** Composite figure (6-panel line plots) titled “RQ3 H3.1 — Vertical vorticity at multiple levels vs stability”. Each panel shows a single series (“output”) of maximum vertical vorticity at a specified height level versus time (hours). All series show rapid spin-up during the first ~50–70 h, followed by variable behavior (plateaus, fluctuations, and some mid-simulation weakening) depending on height.

Top-left: “Max vorticity sfc (1/s)”. Starts near 0, increases steadily to ~0.005 by ~35–45 h, then jumps to ~0.010–0.015 by ~50–70 h with spikes up to ~0.018–0.019 around ~55–65 h. After ~80–120 h decreases to ~0.008–0.011, then recovers modestly to ~0.010–0.013 by ~150–195 h.

Top-middle: “Max vorticity 1 km (1/s)”. Rises from near 0 to ~0.005 by ~35–45 h, then to ~0.010–0.015 by ~50–80 h with intermittent spikes approaching ~0.017–0.018. Shows a dip around ~95–120 h down to ~0.008–0.011, then becomes highly spiky again ~120–190 h, with frequent peaks ~0.015–0.018; ends near ~0.014–0.016.

Top-right: “Max vorticity 2 km (1/s)”. Increases from near 0 to ~0.003–0.005 by ~40–50 h, then to ~0.009–0.011 by ~55–80 h. Around ~105–130 h decreases to ~0.006–0.008, then returns to ~0.009–0.011 during ~140–190 h; ends near ~0.010.

Bottom-left: “Max vorticity 3 km (1/s)”. Gradual increase to ~0.004 by ~40–50 h, then to ~0.007–0.010 by ~55–75 h. Contains a narrow spike near ~55–60 h reaching ~0.017. After ~80–130 h weakens to ~0.006–0.008, then slowly recovers to ~0.007–0.009 by ~160–195 h.

Bottom-middle: “Max vorticity 4 km (1/s)”. Rises from near 0 to ~0.004 by ~35–45 h; sharp increase near ~50 h to ~0.009–0.011 with peaks ~0.012 around ~60–70 h. Then a general decline/plateau ~70–120 h to ~0.006–0.008, with a minimum around ~125–135 h near ~0.005. Later (~140–195 h) modest recovery to ~0.006–0.008.

Bottom-right: “Max vorticity 5 km (1/s)”. Increases to ~0.004 by ~35–45 h; peaks near ~55–70 h around ~0.010–0.013. Then declines to ~0.005–0.007 by ~110–140 h, with intermittent spikes. Ends ~0.0055–0.0065 by ~190–195 h.

**X-axis:** Time (h), 0–200

**Y-axis:** Max vorticity (1/s), varies by level: ~0–0.019 (sfc/1 km), ~0–0.014 (2 km/5 km), ~0–0.0175 (3 km), ~0–0.0125 (4 km)

**Legend:**
- output — blue solid

### 20. `lholk8y` — _plot_

![lholk8y](https://s.odsi.io/lholk8y)

**Caption:** RQ3 H3.1 — Pressure diagnostics vs stability

**Description:** Composite figure (3-panel line plots) titled “RQ3 H3.1 — Pressure diagnostics vs stability”. Single series (“output”) in each panel, plotted versus time (hours).

Left panel: line_plot of “Min sfc pressure (hPa)”. Starts around ~1008–1010 hPa at 0 h, decreases steadily with time. Around ~40–60 h there is a sharper drop from ~990 hPa to ~950 hPa. Between ~60–120 h continues downward with fluctuations, roughly ~930–945 hPa, including a small rebound near ~110–120 h (~935–940). After ~120 h declines further to ~910–920 hPa by ~140–160 h. Reaches the lowest values near the end, around ~900–905 hPa by ~185–195 h.

Middle panel: line_plot of “Max sfc pressure (hPa)”. Values are near ~1014.8–1015.2 hPa early, gradually increasing slightly to ~1015.2–1015.6 by ~100–130 h with noise. A distinct short-lived spike occurs near ~130–135 h reaching ~1016.9 hPa, then drops back to ~1015.0–1015.5. Late times (~150–195 h) fluctuate around ~1014.8–1015.6, with another smaller peak near ~185–190 h (~1015.7–1015.8).

Right panel: line_plot of “Pressure deficit (hPa)”. Starts near ~5–10 hPa, increases gradually to ~20 hPa by ~30–40 h. A rapid rise occurs near ~45–60 h, reaching ~60–70 hPa. Continues increasing to ~80–90 hPa by ~80–100 h, briefly dips around ~110–120 h to ~75–80 hPa, then rises again. After ~130–170 h climbs to ~100–110 hPa, ending near ~110–118 hPa by ~185–195 h.

**X-axis:** Time (h), 0–200

**Y-axis:** Min sfc pressure (hPa) ~900–1010; Max sfc pressure (hPa) ~1014.7–1017.0; Pressure deficit (hPa) ~0–120

**Legend:**
- output — blue solid

### 21. `mccwv9s` — _plot_

![mccwv9s](https://s.odsi.io/mccwv9s)

**Caption:** RQ3 H3.1 — Convective metrics vs stability

**Description:** Composite figure (6-panel line plots) titled “RQ3 H3.1 — Convective metrics vs stability”. All panels show a single time series (“output”) versus time (hours).

Top-left: “Max w (m/s)”. Starts near 0, quickly rises by ~5–10 h to ~5–10 m/s with strong variability. Through ~10–120 h fluctuates mostly ~4–9 m/s with intermittent spikes above ~10–14 m/s (notably near ~55–65 h). A standout spike near ~90 h reaches ~19 m/s. After ~120 h, values trend slightly higher with frequent peaks ~8–12 m/s; ends around ~9–12 m/s.

Top-middle: “Min w (m/s)”. Early (first ~20 h) has strong negative excursions ~−2.5 to ~−4 m/s. From ~25–160 h generally hovers between ~−1.0 and ~−2.5 m/s with occasional sharp dips to ~−3 to ~−4 m/s. Late times (~170–195 h) become more negative again at times, reaching ~−3 to ~−3.5 m/s.

Top-right: “Height of max w (km)”. Starts near 0 km, then quickly jumps within ~5–15 h to variable heights with tall spikes up to ~20–22 km early. After ~20 h, mostly stays between ~3 and ~8 km but with intermittent narrow spikes to ~10–20 km (notable spikes near ~40–60 h and around ~120–130 h). Late times remain mostly ~4–8 km.

Bottom-left: “Cloud top (km)”. Rises rapidly from near 0 to ~15–18 km by ~5–10 h. Thereafter remains high, mostly ~15–18 km, with mild downward drift and reduced variability after ~70 h (often ~15.5–16.5 km). Ends near ~16 km.

Bottom-middle: “Cloud base (km)”. Rapid early rise to ~0.55–0.60 km by ~5–10 h. From ~10–60 h varies between ~0.28 and ~0.42 km with frequent step-like changes. After ~60 h tends to lower values ~0.25–0.30 km, with occasional brief dips to ~0.17–0.20 km (several episodes around ~70–110 h and ~140–190 h).

Bottom-right: “Cloud depth (km)”. Increases rapidly to ~15–18 km by ~5–10 h, then stays mostly ~15–17 km with modest variability and a slight decreasing tendency after ~50–70 h. Ends near ~15.5–16 km.

**X-axis:** Time (h), 0–200

**Y-axis:** Varies by panel: Max w (m/s) ~0–20; Min w (m/s) ~0 to −4.5; Height of max w (km) ~0–22; Cloud top (km) ~0–20; Cloud base (km) ~0–0.6; Cloud depth (km) ~0–19

**Legend:**
- output — blue solid

### 22. `p28t3m3` — _plot_

![p28t3m3](https://s.odsi.io/p28t3m3)

**Caption:** RQ3 H3.1 — Theta-e and moisture vs stability

**Description:** Composite figure (4-panel line plots) titled “RQ3 H3.1 — Theta-e and moisture vs stability”. Single series (“output”) in each panel versus time (hours).

Top-left: “Max theta-e < 10 km (K)”. Starts near ~360–365 K at 0–5 h, then declines to ~359–361 K by ~20–35 h. After ~40 h increases gradually: ~362–368 K by ~60–90 h, then continues rising in steps to ~370–376 K by ~110–140 h. Late period (~140–195 h) climbs further to ~380–384 K, with a brief dip near ~170–175 h (~378–380 K), ending near ~384–385 K.

Top-right: “Max theta-e at lowest level (K)”. Very similar evolution to the top-left panel: early peak ~365 K, early decline to ~359–361 K by ~30 h, then steady rise. Reaches ~370–376 K around ~120–150 h, and ~380–384 K by ~160–195 h, with a small dip near ~170–175 h.

Bottom-left: “Max qv (g/kg)”. Starts near ~20.2 g/kg, rises quickly to ~21.6–21.9 g/kg by ~5–10 h, then drops to ~20.4–20.9 g/kg by ~15–35 h (minimum near ~20.3). Gradual recovery to ~20.9–21.1 g/kg by ~50–75 h, then a slight dip around ~80–100 h (~20.5–20.8). From ~105 h onward increases more strongly, reaching ~21.5–22.5 g/kg by ~120–140 h, then ~22.8–23.6 g/kg by ~150–170 h. Ends near ~23.0–24.0 g/kg with the highest value close to ~24 g/kg at ~190–195 h.

Bottom-right: “Min theta-e at lowest level (K)”. Starts near ~353–354 K, then rapidly drops within the first ~10–20 h to ~339–342 K (lowest around ~338–339 K). From ~20–60 h rebounds to ~345–351 K with strong variability. Between ~60–120 h fluctuates mostly ~345–349 K, with a sharp dip near ~110–115 h to ~340 K. After ~120 h continues oscillating ~342–347 K with repeated dips; ends around ~343–345 K.

**X-axis:** Time (h), 0–200

**Y-axis:** Max theta-e (K) ~358–385; Max qv (g/kg) ~20.2–24.0; Min theta-e (K) ~338–354

**Legend:**
- output — blue solid

### 23. `l5cqibn` — _plot_

![l5cqibn](https://s.odsi.io/l5cqibn)

**Caption:** RQ3 H3.1 — Precipitation and mass flux vs stability

**Description:** Composite figure (3-panel line plots) titled “RQ3 H3.1 — Precipitation and mass flux vs stability”. Each panel shows a single series (“output”) versus time (hours).

Left: line_plot of “Max precip rate (kg/m^2/s)”. Near 0 at the beginning, rises quickly by ~5–10 h to ~0.015–0.020. Between ~10–40 h decreases and fluctuates mostly ~0.006–0.015 with occasional dips near ~0.003–0.005. Around ~45–60 h shifts upward to ~0.020–0.035 with frequent spikes; thereafter remains generally elevated, typically ~0.018–0.032 through ~60–195 h with intermittent peaks ~0.034–0.037 (largest near the end).

Middle: line_plot of “Total upward mass flux (kg m/s)” with a 1e13 scale indicated on the axis (values are ×10^13). Increases from near 0 to ~0.25–0.35×10^13 by ~20–30 h, then gradually rises to ~0.5–0.8×10^13 by ~70–110 h. Around ~120–135 h reaches ~1.0–1.2×10^13 followed by a sharp transient spike near ~130–135 h up to ~3.8–3.9×10^13. Immediately after, drops back to ~0.8–1.0×10^13 and declines to ~0.5–0.7×10^13 by ~150–170 h. Near ~185–190 h shows another bump to ~1.6–1.7×10^13, ending around ~0.8–0.9×10^13.

Right: line_plot of “Total downward mass flux (kg m/s)” also with a 1e13 scale (values ×10^13) and negative values. Starts near 0 then decreases to about ~−0.3 to −0.5×10^13 by ~10–30 h. Gradually becomes more negative to ~−0.8 to −1.1×10^13 by ~90–120 h. Exhibits a very large negative spike near ~130–135 h reaching ~−3.9×10^13, then recovers to about ~−0.6 to −1.0×10^13 afterward. Another negative excursion occurs near ~185–190 h reaching roughly ~−1.6 to −1.8×10^13. Ends around ~−0.9×10^13.

**X-axis:** Time (h), 0–200

**Y-axis:** Max precip rate (kg/m^2/s) ~0–0.038; Total upward mass flux (kg m/s) ~0–4.0×10^13; Total downward mass flux (kg m/s) ~0 to −4.0×10^13

**Legend:**
- output — blue solid

### 24. `svdb2h6` — _plot_

![svdb2h6](https://s.odsi.io/svdb2h6)

**Caption:** RQ3 H3.1 — RMW vs max wind (scatter)

**Description:** Scatter_plot titled “RQ3 H3.1 — RMW vs max wind (scatter)”. Points (blue circular markers) show maximum wind speed (y) versus radius of maximum wind, RMW (x).

Axes ranges visible: RMW from ~0 to ~225 km; Max wind from ~10 to ~110 m/s.

Point distribution shows two main regimes:
- A dense vertical cluster at low RMW (~15–30 km) with high max winds, mostly ~70–105 m/s. Many points around RMW ~25–27 km span roughly ~85–110 m/s, indicating high intensity at relatively small RMW.
- A sparse set of points at larger RMW (~35–220 km) with much lower max winds, mostly ~12–35 m/s. Several outliers at very large RMW (~190–220 km) have max wind around ~22–26 m/s.

There is an overall negative association (higher winds occur almost exclusively at smaller RMW; larger RMW corresponds to weaker winds), though within the small-RMW cluster winds vary widely.

Notable low-wind points at small RMW also exist (e.g., near RMW ~20–30 km with max wind ~20–45 m/s), suggesting early/weak stages overlapping in RMW with intense stages.

**X-axis:** RMW (km), ~0–225

**Y-axis:** Max wind (m/s), ~10–110

**Legend:**
- output — blue circle markers

### 25. `5dgfbyu` — _plot_

![5dgfbyu](https://s.odsi.io/5dgfbyu)

**Caption:** RQ3 H3.1 — Intensity change rate vs stability

**Description:** Composite figure (2-panel line plots) titled “RQ3 H3.1 — Intensity change rate vs stability”. Both panels show time derivatives versus time (hours), with a dashed horizontal reference line at 0.

Left panel: line_plot of “dVmax/dt (m/s per hr)” vs time. Series oscillates frequently around 0 with high variability. Early times (~0–30 h) range roughly from ~−5 to +5. Between ~40–80 h contains several strong positive spikes up to ~+10 to +13 and negative spikes down to about ~−11. From ~80–160 h continues oscillatory behavior, typically within ~−6 to +7, with occasional positive spikes near ~+10–12 (e.g., around ~120–140 h). Late times (~160–195 h) still variable, with a few negative excursions near ~−6 to −8 and positive excursions up to ~+5; ends slightly positive (~+1 to +2).

Right panel: line_plot of “dPmin/dt (hPa/hr)” vs time. Also oscillates around 0 (dashed line). Many negative values (pressure falling) and positive bursts (pressure rising). Typical fluctuations are about ~−4 to +4, with several deep negative spikes: around ~55–70 h to roughly ~−8 to −9, and a later very deep minimum near ~140–150 h reaching about ~−11 hPa/hr. Positive spikes reach about ~+5 to +6 (e.g., around ~70–100 h and again ~145–155 h). Ends near ~0 to slightly negative.

**X-axis:** Time (h), 0–200

**Y-axis:** dVmax/dt (m/s per hr) ~−12 to +13; dPmin/dt (hPa/hr) ~−11 to +6

**Legend:**
- output — blue solid

### 26. `fv5l9i9` — _plot_

![fv5l9i9](https://s.odsi.io/fv5l9i9)

**Caption:** RQ3 H3.1 — Summary bar charts (peak metrics)

**Description:** Composite figure (4-panel bar charts) titled “RQ3 H3.1 — Summary bar charts (peak metrics)”. Each panel shows a single bar for category “output”.

Panel 1 (left): bar_chart of “Peak Vmax (m/s)”. Single bar reaches approximately ~108–110 m/s.

Panel 2: bar_chart of “Min Psfc (hPa)”. Single bar reaches approximately ~900 hPa (near the 900 mark).

Panel 3: bar_chart of “Peak Wmax (m/s)”. Single bar reaches approximately ~19 m/s.

Panel 4 (right): bar_chart of “Min RMW (km)”. Single bar reaches approximately ~14 km.

With only one category shown, no within-figure comparison across categories is possible; the panels summarize four different peak metrics for the same labeled output.

**X-axis:** Category (single): output

**Y-axis:** Varies by panel: Peak Vmax (m/s) ~0–110; Min Psfc (hPa) ~0–900; Peak Wmax (m/s) ~0–20; Min RMW (km) ~0–14

### 27. `2k8fxjn` — _plot_

![2k8fxjn](https://s.odsi.io/2k8fxjn)

**Caption:** RQ3 H3.1 — Intensity and structure vs environmental stability

**Description:** Composite figure (4-panel line plots; subtype: line_plot time series) titled “RQ3 H3.1 — Intensity and structure vs environmental stability”. All panels show a single blue solid line labeled “output”.

Top-left (Max horiz. wind (m/s) vs Time): Starts near ~14–15 m/s at 0 h, with an early spike to ~28–30 m/s around ~10 h. From ~20–30 h it is ~18–22 m/s, then intensifies strongly: ~20→~80 m/s from ~30 to ~75 h. It then fluctuates around ~80–90 m/s through ~150 h with a noticeable dip to ~78–82 m/s around ~115–140 h. A second strengthening occurs after ~150 h, reaching ~95–100 m/s by ~160–175 h, peaking near ~102–104 m/s around ~165–170 h, then easing to ~90–93 m/s by ~190–195 h.

Top-right (Min sfc pressure (hPa) vs Time): Begins around ~1008–1010 hPa (0–30 h), then decreases steadily: ~1000→~980 hPa by ~40–50 h, ~965 hPa by ~60 h, and ~940 hPa by ~75–85 h. Between ~85 and ~150 h it oscillates roughly ~927–942 hPa (non-monotonic). After ~150 h it drops again to ~920–923 hPa by ~155–160 h, reaching a minimum around ~913–915 hPa near ~170 h, then rebounds to ~925–927 hPa by ~190–195 h.

Bottom-left (RMW (km) vs Time): Early times show very large and erratic values: ~75–85 km at ~0–5 h, then an abrupt collapse to ~2–5 km by ~10–15 h, with brief spikes (e.g., ~55 km near ~25–30 h). From ~30–70 h it remains mostly low (~5–15 km) with step-like segments. After ~70 h it becomes more “stair-stepped”: ~14 km plateau (~70–120 h), ~18 km (~120–145 h), then ~21–22 km (~145–195 h) with occasional short dips.

Bottom-right (Max vert. velocity (m/s) vs Time): Near 0 at the start, then a sharp peak around ~27–28 m/s near ~10 h. From ~10–60 h it fluctuates mostly ~4–7 m/s with intermittent spikes to ~10–16 m/s. From ~60–150 h it gradually trends upward to ~7–10 m/s with continued noise. Around ~155–175 h it rises to a higher regime (~10–14 m/s, with multiple points near ~12–14), then declines toward ~7–9 m/s by ~190–195 h.

**X-axis:** Time (h), 0–200

**Y-axis:** Multiple y-axes by panel: Max horiz. wind (m/s) ~10–105; Min sfc pressure (hPa) ~910–1015; RMW (km) ~0–90; Max vert. velocity (m/s) ~0–28

**Notes:** RMW series shows extreme early spikes and strong stair-step/quantized behavior later; cloud/structure-like diagnostics may be discretized or intermittently undefined. Despite “vs environmental stability” framing, only a single series (“output”) is shown in each panel.

**Legend:**
- output — blue solid

### 28. `puf47i7` — _plot_

![puf47i7](https://s.odsi.io/puf47i7)

**Caption:** RQ3 H3.1 — 10 m wind and convective depth

**Description:** Two-panel line-plot time series titled “RQ3 H3.1 — 10 m wind and convective depth”. Each panel has one blue solid curve labeled “output”.

Left (Max 10 m wind (m/s) vs Time): Starts ~12–13 m/s at 0 h with a brief dip near ~10–11 m/s early, then an early spike to ~21–22 m/s around ~10 h. After ~20 h it increases steadily: ~18–20 m/s (~20–25 h), ~30 m/s (~35–40 h), ~40 m/s (~50–55 h), ~50 m/s (~60–65 h), ~60 m/s (~75–85 h), reaching ~67–69 m/s around ~100–110 h. It weakens to ~58–62 m/s around ~120–145 h, then re-intensifies after ~150 h to ~70–75 m/s around ~160–175 h (peak ~75–76 near ~168–172 h), ending around ~66–68 m/s by ~190–195 h.

Right (Cloud top height (km) vs Time): Near ~0 km at the beginning, then a rapid jump to deep convection by ~8–12 h, reaching ~15–16 km. Peaks around ~19–20 km near ~15–25 h. Thereafter it fluctuates mostly ~16–18 km through ~60–70 h, then settles into a lower, more step-like regime around ~15–16 km from ~80 h onward with small discrete jumps; ends near ~15.5–16 km by ~190–195 h.

**X-axis:** Time (h), 0–200

**Y-axis:** Left: Max 10 m wind (m/s) ~10–76; Right: Cloud top height (km) ~0–20

**Notes:** Cloud-top height becomes notably quantized/stair-stepped after ~80 h, suggesting discretization or a threshold-based diagnostic.

**Legend:**
- output — blue solid

### 29. `1ylfoin` — _plot_

![1ylfoin](https://s.odsi.io/1ylfoin)

**Caption:** RQ3 H3.1 — Theta-e and surface vorticity

**Description:** Two-panel line-plot time series titled “RQ3 H3.1 — Theta-e and surface vorticity”. Both panels show a single blue solid line labeled “output”.

Left (Max theta-e below 10 km (K) vs Time): Starts near ~360–361 K at 0 h, rises quickly to ~366 K by ~3–6 h, then drops to ~362.5–363 K around ~15–20 h. From ~20–60 h it slowly climbs and becomes noisy (~363–369 K) with a notable spike to ~373–374 K near ~60 h. Between ~60–120 h it fluctuates mostly ~366–371 K with intermittent spikes and dips. A dip occurs near ~120–130 h to ~365–366 K, followed by a sustained increase after ~140–150 h to ~373–375 K by ~160–170 h. Late-time peaks reach ~378–379 K around ~180–185 h, then it fluctuates ~373–377 K toward ~190–195 h.

Right (Max sfc vorticity (1/s) vs Time): Begins very small (~0.0003–0.001) and increases gradually to ~0.002 by ~25–30 h. A sharp intensification occurs around ~35–45 h to ~0.008–0.009. It remains ~0.007–0.010 through ~70 h, then rises again around ~70–90 h into ~0.015–0.019. Peak values are near ~0.020–0.021 around ~95–110 h. After ~110–120 h it decreases and stabilizes around ~0.011–0.014 with intermittent spikes (~0.015–0.017) through ~190–195 h.

**X-axis:** Time (h), 0–200

**Y-axis:** Left: Max theta-e below 10 km (K) ~360–379; Right: Max sfc vorticity (1/s) ~0–0.021

**Notes:** Despite “vs stability” framing, only one time series (“output”) is plotted per panel (no stable/unstable comparison visible).

**Legend:**
- output — blue solid

### 30. `7brja2t` — _plot_

![7brja2t](https://s.odsi.io/7brja2t)

**Caption:** RQ3 H3.1 — BL wind and PBL depth vs stability

**Description:** Two-panel line-plot time series titled “RQ3 H3.1 — BL wind and PBL depth vs stability”. Both panels show a single blue solid line labeled “output”.

Left (Max horiz. wind at lowest level (m/s) vs Time): Starts ~12–13 m/s at 0 h with a brief dip near ~10–11 m/s, then an early spike to ~23–24 m/s around ~10 h. From ~20–75 h it intensifies steadily from ~18–20 m/s to ~60–65 m/s. It reaches ~73–76 m/s around ~100–110 h, weakens to ~64–70 m/s around ~120–145 h, then re-strengthens to ~80–85 m/s near ~160–175 h (peak ~84–86), ending around ~74–76 m/s by ~190–195 h.

Right (Max PBL depth (km) vs Time): Starts around ~0.55–0.7 km and rises to ~0.9–1.1 km by ~10–20 h. It increases to ~1.2–1.3 km around ~30–40 h, then climbs more rapidly to ~1.5–1.7 km by ~50–60 h and ~2.0–2.3 km by ~70–80 h. From ~80–150 h it fluctuates broadly ~2.1–2.7 km. It reaches its highest values near ~2.8–3.0 km around ~160–175 h, then declines modestly to ~2.3–2.5 km by ~190–195 h.

**X-axis:** Time (h), 0–200

**Y-axis:** Left: Max horiz. wind at lowest level (m/s) ~10–86; Right: Max PBL depth (km) ~0.5–3.0

**Notes:** PBL depth shows regime shifts and noisy variability; no multi-case stability comparison is visible (single “output” series).

**Legend:**
- output — blue solid

### 31. `if5vhdn` — _plot_

![if5vhdn](https://s.odsi.io/if5vhdn)

**Caption:** RQ3 H3.1 — Lowest-model-level wind components vs stability

**Description:** Composite 2×2 figure (line_plot time series) titled “RQ3 H3.1 — Lowest-model-level wind components vs stability”. All panels show a single blue solid line labeled “output”.

Top-left (Max u at lowest level (m/s) vs Time): Near 0 at the start, then rises sharply to a peak ~13–14 m/s around ~12–18 h. Thereafter it declines into a noisy mid-range (~4–8 m/s) through ~20–60 h, with intermittent spikes. Around ~65–75 h it has another burst with spikes near ~10–11 m/s, then drops abruptly toward ~0–1 m/s around ~80–90 h. From ~90–200 h it stays mostly low (~1–3 m/s) with occasional spikes up to ~6–8 m/s (notably around ~110–130 h), ending ~2–4 m/s.

Top-right (Min u at lowest level (m/s) vs Time): Starts near 0 and quickly becomes negative, reaching ~−10 to −12 m/s by ~10–15 h. From ~15–60 h it fluctuates roughly ~−8 to −14 m/s. A marked shift to stronger negative values occurs near ~65–80 h, dropping to ~−25 to −35 m/s. From ~80–150 h it remains mostly ~−28 to −36 m/s with noise and brief dips. After ~150 h it becomes even more negative, reaching ~−42 to −48 m/s around ~160–175 h, then rebounds to about ~−34 to −36 m/s by ~190–195 h.

Bottom-left (Max v at lowest level (m/s) vs Time): Starts around ~12–15 m/s, increases to ~18–22 m/s by ~15–25 h, then rises steadily to ~40 m/s by ~50–55 h and ~60–65 m/s by ~70–85 h. Peaks around ~70–72 m/s near ~100–110 h, dips to ~61–66 m/s around ~120–145 h, then re-peaks near ~77–79 m/s around ~160–175 h, ending ~71–73 m/s by ~190–195 h.

Bottom-right (Min v at lowest level (m/s) vs Time): Near 0 (slightly negative) through ~60–65 h, then an abrupt onset of more negative excursions around ~70–120 h with frequent dips reaching ~−8 to ~−12 m/s (most values between ~−2 and ~−8). After ~120–140 h it relaxes toward weaker negatives (~−0.5 to −2 m/s), with occasional dips to ~−3 to −4 m/s. Ends near ~−1 to −2 m/s.

**X-axis:** Time (h), 0–200

**Y-axis:** Multiple y-axes by panel: Max u (m/s) ~0–14; Min u (m/s) ~0 to −50; Max v (m/s) ~10–80; Min v (m/s) ~0 to −12

**Notes:** Min v remains ~0 for a long initial period then shows abrupt large negative excursions after ~70 h, suggesting a regime change or threshold/diagnostic behavior. Single-series plotting does not show separate stability cases.

**Legend:**
- output — blue solid

### 32. `srz522z` — _plot_

![srz522z](https://s.odsi.io/srz522z)

**Caption:** RQ3 H3.1 — Vertical vorticity at multiple levels vs stability

**Description:** Composite 2×3 figure (six line_plot time series) titled “RQ3 H3.1 — Vertical vorticity at multiple levels vs stability”. Each panel shows a single blue solid line labeled “output”. X-axis is Time (h) in all panels.

Top row:
• Max vorticity sfc (1/s): Starts ~0.0003–0.001, rises gradually to ~0.002 by ~25–30 h, then jumps to ~0.008–0.010 around ~35–45 h. It increases further to ~0.016–0.019 by ~70–100 h, with peak values slightly above ~0.020 around ~90–110 h. After ~110–120 h it decreases and stabilizes around ~0.011–0.014 through ~190–195 h.
• Max vorticity 1 km (1/s): Similar early evolution (small → ~0.002 by ~30 h), then a jump to ~0.010–0.012 by ~35–45 h. It trends upward to ~0.015–0.018 by ~70–100 h with intermittent spikes near ~0.020–0.021 around ~90–120 h. Later it fluctuates mostly ~0.011–0.015, with some late dips near ~0.009–0.010 toward ~190 h.
• Max vorticity 2 km (1/s): Starts near ~0.0003–0.001, rises to ~0.002 by ~25–30 h, then steps up to ~0.009–0.012 around ~35–45 h. Peaks around ~0.013–0.015 near ~70–100 h. Decreases to ~0.008–0.010 by ~120–150 h, then modestly recovers to ~0.010–0.011 toward ~170–195 h.

Bottom row:
• Max vorticity 3 km (1/s): Rises from ~0.0003–0.001 to ~0.002 by ~25–30 h, then increases sharply to ~0.009–0.012 near ~40–50 h. Holds around ~0.009–0.011 through ~100 h, then declines to a minimum near ~0.006 around ~145–155 h, and partially recovers to ~0.008–0.010 by ~170–195 h.
• Max vorticity 4 km (1/s): Similar early ramp and jump near ~40–50 h to ~0.008–0.013; peaks near ~0.013–0.014 around ~60–85 h. It then decreases toward ~0.006 around ~120–155 h and fluctuates ~0.006–0.008 thereafter, with an uptick close to ~0.009 by ~190–195 h.
• Max vorticity 5 km (1/s): Increases from near-zero to ~0.002–0.003 by ~30 h, then jumps to ~0.007–0.010 near ~40–60 h. Shows spikes around ~0.013–0.014 near ~70–90 h, then settles lower (~0.0055–0.007) through ~120–170 h, ending with a slight rise (~0.008–0.010) near ~190–195 h.

**X-axis:** Time (h), 0–200

**Y-axis:** Multiple y-axes by panel: Max vorticity (1/s) ranges from ~0–0.021 depending on level (sfc/1 km highest, 3–5 km generally lower)

**Notes:** All panels show only one series (“output”), so no explicit stable/unstable separation is visible despite the title.

**Legend:**
- output — blue solid

### 33. `bi0b3pd` — _plot_

![bi0b3pd](https://s.odsi.io/bi0b3pd)

**Caption:** RQ3 H3.1 — Pressure diagnostics vs stability

**Description:** Three-panel line-plot time series titled “RQ3 H3.1 — Pressure diagnostics vs stability”. Each panel shows a single blue solid line labeled “output”.

Left (Min sfc pressure (hPa) vs Time): Starts around ~1008–1010 hPa (0–30 h), then falls to ~980 hPa by ~45–50 h, ~960–965 hPa by ~60 h, and ~940 hPa by ~75–85 h. From ~85–150 h it oscillates broadly (~927–942 hPa). After ~150 h it drops again to ~920–923 hPa by ~155–160 h, reaches a minimum near ~913–915 hPa around ~170 h, then rebounds to ~925–927 hPa by ~190–195 h.

Middle (Max sfc pressure (hPa) vs Time): Narrow-range variability around ~1014.8–1015.6 hPa. Begins near ~1014.80 hPa, rises toward ~1015.0 by ~15–25 h, then fluctuates around ~1014.9–1015.1. A prominent spike occurs near ~70–80 h reaching ~1015.5–1015.6 hPa. Thereafter it remains noisy mostly ~1014.85–1015.05 with intermittent small spikes up to ~1015.1–1015.2 through ~190–195 h.

Right (Pressure deficit (hPa) vs Time): Starts small (~5–10 hPa) and increases steadily: ~20–30 hPa by ~30–40 h, ~40–55 hPa by ~50–60 h, and ~70–80 hPa by ~80–110 h. It fluctuates around ~70–85 hPa between ~100–150 h, then increases again after ~150 h to ~95–102 hPa around ~160–175 h, ending near ~88–92 hPa by ~190–195 h.

**X-axis:** Time (h), 0–200

**Y-axis:** Left: Min sfc pressure (hPa) ~910–1015; Middle: Max sfc pressure (hPa) ~1014.8–1015.6; Right: Pressure deficit (hPa) ~0–105

**Notes:** Middle panel’s y-range is extremely narrow relative to left/right; the large spike near ~70–80 h (to ~1015.6 hPa) stands out versus surrounding variability.

**Legend:**
- output — blue solid

### 34. `bvq7ek1` — _plot_

![bvq7ek1](https://s.odsi.io/bvq7ek1)

**Caption:** RQ3 H3.1 — Convective metrics vs stability

**Description:** Composite 2×3 figure (six line_plot time series) titled “RQ3 H3.1 — Convective metrics vs stability”. Each panel shows a single blue solid line labeled “output”.

Top-left (Max w (m/s) vs Time): Near 0 initially, then a sharp peak around ~27–28 m/s near ~10 h. Afterward it is mostly ~4–8 m/s with frequent spikes to ~10–16 m/s (especially ~10–40 h). From ~60–150 h it trends slightly upward (~6–10 m/s). Around ~155–175 h it reaches a higher regime (~10–14 m/s), then drops back toward ~7–9 m/s by ~190–195 h.

Top-middle (Min w (m/s) vs Time): Near 0 at the very beginning, then quickly becomes negative (~−2 to −3.5 m/s). Shows intermittent deeper downdraft excursions to ~−5 to ~−6 m/s (notably around ~30–40 h and again near ~160 h). Many values remain clustered around ~−1.5 to −3 m/s through the rest of the simulation, ending around ~−3 to −4 m/s.

Top-right (Height of max w (km) vs Time): Rises from ~0 to ~4–6 km by ~10 h, then highly variable with frequent tall spikes reaching ~18–21 km during ~15–55 h. From ~60–140 h it is mostly ~4–7 km, with at least one prominent spike back up near ~20 km around ~120–130 h. After ~150 h it trends upward to ~6–9 km and ends near ~9–11 km by ~190–195 h.

Bottom-left (Cloud top (km) vs Time): Rapid deepening from ~0 to ~15–16 km by ~10–12 h, peak ~19–20 km near ~15–25 h, then fluctuates ~16–18 km through ~60–70 h. After ~80 h it becomes step-like around ~15–16 km, ending near ~15.5–16 km.

Bottom-middle (Cloud base (km) vs Time): Starts at ~0, then quickly rises to ~0.4–0.45 km early, with brief excursions up to ~0.55 km around ~10–20 h. Over time it alternates between discrete plateaus around ~0.28 km and ~0.41 km, with periods near ~0.17–0.20 km around ~85–110 h and intermittent dips again around ~140–160 h. Ends near ~0.28 km.

Bottom-right (Cloud depth (km) vs Time): Increases rapidly to ~15–16 km by ~10–12 h, peaks near ~19–20 km around ~15–25 h, then decreases to ~15–17 km and becomes more step-like after ~80 h (generally ~15–16 km), ending near ~15.5–16 km.

**X-axis:** Time (h), 0–200

**Y-axis:** Multiple y-axes by panel: w (m/s) ~−6 to 28; Height of max w (km) ~0–21; Cloud top (km) ~0–20; Cloud base (km) ~0–0.55; Cloud depth (km) ~0–20

**Notes:** Several cloud/height diagnostics show strong discretization (step-like plateaus, repeated discrete levels), suggesting thresholded or binned diagnostic outputs.

**Legend:**
- output — blue solid

### 35. `ihtcexl` — _plot_

![ihtcexl](https://s.odsi.io/ihtcexl)

**Caption:** RQ3 H3.1 — Theta-e and moisture vs stability

**Description:** Composite 2×2 figure (line_plot time series) titled “RQ3 H3.1 — Theta-e and moisture vs stability”. All panels show a single blue solid line labeled “output”.

Top-left (Max theta-e < 10 km (K) vs Time): Starts near ~360–361 K, rises quickly to ~366 K by ~3–6 h, drops to ~362.5–363 K around ~15–20 h, then gradually increases with strong noise and intermittent spikes through ~120 h (notably reaching ~373–374 K near ~60 h). A dip occurs near ~120–130 h (~365–366 K), followed by a late rise after ~140–150 h reaching ~373–375 K by ~160–170 h and peaking around ~378–379 K near ~180–185 h; ends fluctuating ~373–377 K.

Top-right (Max theta-e at lowest level (K) vs Time): Very similar evolution and range to the top-left panel (rapid early rise to ~366 K, mid-run noisy ~364–371 K, late increase to mid/high ~370s K, peak near ~378–379 K around ~180–185 h).

Bottom-left (Max qv (g/kg) vs Time): Starts near ~20.3 g/kg, rises rapidly to ~22.2–22.3 g/kg by ~5–10 h, then drops to ~21.1–21.2 g/kg around ~15–25 h. From ~25–60 h it fluctuates ~21.1–21.6 g/kg with brief spikes; a prominent spike reaches ~23.1–23.2 g/kg near ~60 h. From ~60–120 h it trends upward to ~22.2–22.4 g/kg around ~90–100 h, then dips to ~21.7–22.0 g/kg near ~115–130 h. It rises again to ~22.5–22.7 g/kg around ~140–155 h, then eases to ~22.1–22.3 g/kg by ~165–175 h with a couple late spikes (~22.7–22.9 g/kg) near ~175–190 h, ending around ~21.8–22.2 g/kg.

Bottom-right (Min theta-e at lowest level (K) vs Time): Starts near ~353.5 K, then after ~10 h drops into a noisy lower regime (~348–351 K) through ~40–50 h. It rises toward ~352–355 K by ~55–70 h, then shows a sharp minimum near ~344 K around ~75 h. After that it rebounds strongly to ~354–356 K by ~90–105 h and stays mostly ~353–355 K (noisy) through ~140–150 h, then gradually decreases after ~150–160 h to ~348–350 K by ~170–195 h.

**X-axis:** Time (h), 0–200

**Y-axis:** Multiple y-axes by panel: theta-e (K) ~360–379 (max) and ~344–356 (min); qv (g/kg) ~20.3–23.2

**Notes:** Bottom-right series shows an unusually sharp isolated minimum (~344 K near ~75 h) compared with surrounding values.

**Legend:**
- output — blue solid

### 36. `todrlvt` — _plot_

![todrlvt](https://s.odsi.io/todrlvt)

**Caption:** RQ3 H3.1 — Precipitation and mass flux vs stability

**Description:** Composite figure with three time-series line plots (subtype: line_plot) under the suptitle “RQ3 H3.1 — Precipitation and mass flux vs stability”. All three panels plot a single blue series labeled “output”.

Left panel (max precipitation rate):
- X: Time from 0 to 200 h.
- Y: “Max precip rate (kg/m²/s)” from ~0.00 to ~0.05.
- Series behavior: starts near 0 at t=0, then quickly becomes intermittent with an early sharp spike to ~0.047 around ~10 h. From ~10–60 h fluctuates mostly ~0.005–0.02 with frequent jagged spikes. Around ~70 h it steps up to a higher regime (~0.02–0.035), with occasional peaks approaching ~0.048 (near ~70 h) and multiple spikes ~0.035–0.04 between ~90–115 h. After ~120 h it oscillates around ~0.02–0.03 with sporadic peaks ~0.04 (notably near ~150–170 h). Ends near ~0.025–0.03 by ~195–200 h.

Middle panel (total upward mass flux):
- X: Time from 0 to 200 h.
- Y: “Total upward mass flux (kg m/s)” with scientific-notation scale indicator “1e12”; visible range roughly 0 to 5×10^12.
- Series behavior: near 0 at start, rises rapidly to ~1.5×10^12 by ~15 h, then increases to ~2.5–3.0×10^12 by ~30–45 h. From ~45–130 h it remains highly variable, mostly between ~1.5×10^12 and ~3.2×10^12, with multiple local peaks near ~70–80 h (~3.3×10^12) and dips near ~60–90 h (~1.4–1.8×10^12). Around ~145–155 h a pronounced surge peaks near ~5×10^12 (maximum) and then falls back toward ~3×10^12. Late period (~160–200 h) oscillates broadly ~2.2–4.3×10^12, ending high near ~4.2×10^12.

Right panel (total downward mass flux):
- X: Time from 0 to 200 h.
- Y: “Total downward mass flux (kg m/s)” with “1e12” scale; values are negative, spanning about 0 down to ~−5×10^12.
- Series behavior: begins near 0, drops steeply to about −1.5 to −2×10^12 by ~10–20 h, then continues to more negative values near −2.5 to −3.5×10^12 by ~30–60 h. From ~60–140 h it oscillates between roughly −1.7×10^12 and −4.2×10^12 with repeated dips. A deepest trough occurs near ~145 h reaching approximately −5.2×10^12. After ~150 h it rebounds somewhat (less negative) then remains variable (~−2.5 to −4.2×10^12), ending near ~−4.4×10^12 by ~195–200 h.

**X-axis:** Left: Time (h), 0–200; Middle: Time (h), 0–200; Right: Time (h), 0–200

**Y-axis:** Left: Max precip rate (kg/m²/s), ~0.00–0.05; Middle: Total upward mass flux (kg m/s), ~0–5×10^12; Right: Total downward mass flux (kg m/s), ~0 to −5×10^12

**Legend:**
- output — blue solid line

### 37. `84h1lr2` — _plot_

![84h1lr2](https://s.odsi.io/84h1lr2)

**Caption:** RQ3 H3.1 — RMW vs max wind (scatter)

**Description:** Single-panel scatter plot (subtype: scatter_plot) titled “RQ3 H3.1 — RMW vs max wind (scatter)”. Points are semi-transparent blue circles labeled “output”.

- X-axis: “RMW (km)”, spanning roughly 0 to 90 km.
- Y-axis: “Max wind (m/s)”, spanning roughly 10 to 110 m/s.

Structure/patterns:
- Main clusters occur at moderate RMW (~10–25 km) with high max winds (~70–105 m/s). Notable dense vertical bands include:
  - Around RMW ~14–16 km: many points between ~70 and ~93 m/s, plus a few down near ~50–60 m/s.
  - Around RMW ~18–20 km: many points around ~77–95 m/s.
  - Around RMW ~22 km: a dense set around ~85–99 m/s, including the highest point near ~104 m/s.
- Smaller-RMW, lower-wind groups:
  - RMW ~0–2 km: points mostly ~18–23 m/s.
  - RMW ~7 km: points around ~27–41 m/s.
  - RMW ~10 km: a broader spread roughly ~23–63 m/s.
- Large-RMW outliers with weak winds:
  - Several isolated points at RMW ~55–88 km with winds ~12–19 m/s.

Overall relationship is non-monotonic: the strongest winds occur at intermediate RMW (~15–25 km), while both very small and very large RMW correspond to weaker winds in this sample.

**X-axis:** RMW (km), ~0–90

**Y-axis:** Max wind (m/s), ~10–110

**Legend:**
- output — blue circular markers

### 38. `k7ny839` — _plot_

![k7ny839](https://s.odsi.io/k7ny839)

**Caption:** RQ3 H3.1 — Intensity change rate vs stability

**Description:** Composite figure with two time-series line plots (subtype: line_plot) under the suptitle “RQ3 H3.1 — Intensity change rate vs stability”. Both panels show a single blue line labeled “output” and include a black dashed horizontal reference line at 0.

Left panel (dVmax/dt):
- X-axis: “Time (h)” from 0 to 200.
- Y-axis: “dVmax/dt (m/s per hr)” ranging approximately from −10 to +15.
- Series behavior: highly noisy, oscillating around 0 throughout. Early period includes a sharp positive spike near ~+14 around ~10 h and a sharp negative excursion near ~−10 around roughly the same early window. Between ~20–90 h there are frequent spikes up to ~+10 to +14 (notably around ~35–40 h and ~70 h) and dips to ~−7 to −9 (around ~50–80 h). From ~100–160 h variability continues with repeated negative dips around ~−6 to −9 (notably ~115–150 h) interspersed with moderate positives up to ~+6 to +8. Late times (~160–200 h) remain near zero on average with smaller oscillations, ending slightly negative to near 0.

Right panel (dPmin/dt):
- X-axis: “Time (h)” from 0 to 200.
- Y-axis: “dPmin/dt (hPa/hr)” ranging approximately from −10 to +6.
- Series behavior: similarly noisy about 0. After near-zero start, it oscillates with repeated negative drops to ~−5 to −8 during ~25–60 h and one deepest drop near ~−10 around ~75 h. Positive spikes reach about +5 to +6 near ~30 h and several times later (~110–140 h). From ~150–200 h the series continues fluctuating between roughly −6 and +3, with the final segment appearing slightly positive (~0 to +2) before ending near ~0 to +1.

**X-axis:** Left: Time (h), 0–200; Right: Time (h), 0–200

**Y-axis:** Left: dVmax/dt (m/s per hr), ~−10–15; Right: dPmin/dt (hPa/hr), ~−10–6

**Legend:**
- output — blue solid line
- 0 reference — black dashed horizontal

### 39. `0u06umz` — _plot_

![0u06umz](https://s.odsi.io/0u06umz)

**Caption:** RQ3 H3.1 — Summary bar charts (peak metrics)

**Description:** Composite figure of four single-bar charts (subtype: bar_chart) titled “RQ3 H3.1 — Summary bar charts (peak metrics)”. Each panel contains one blue bar for the single category “output” (x tick label rotated).

Panel 1 (Peak Vmax):
- Y-axis label: “Peak Vmax (m/s)”.
- Single bar height is just above 100, approximately ~104 m/s.

Panel 2 (Min Psfc):
- Y-axis label: “Min Psfc (hPa)”.
- Single bar height is slightly above 900, approximately ~910 hPa.

Panel 3 (Peak Wmax):
- Y-axis label: “Peak Wmax (m/s)”.
- Single bar height is near the upper 20s, approximately ~28 m/s.

Panel 4 (Min RMW):
- Y-axis label: “Min RMW (km)”.
- Single bar reaches about ~2.0 km.

No error bars, confidence intervals, or multiple categories are shown; each metric is summarized by one value for “output”.

**X-axis:** All panels: category axis with single label “output”

**Y-axis:** Panel 1: Peak Vmax (m/s), ~0–110; Panel 2: Min Psfc (hPa), ~0–950; Panel 3: Peak Wmax (m/s), ~0–30; Panel 4: Min RMW (km), ~0–2.0

**Notes:** Despite the title implying comparison “vs stability”, each subplot shows only a single bar/category (“output”), so no stability-conditioned comparison is visible in this figure.


In [4]:
# Save the Stage-5 Markdown report
if isinstance(analysis_result, ExperimentAnalysisOutputSchema):
    report_path = Path("stage_5_report.md")
    report_path.write_text(analysis_result.markdown, encoding="utf-8")
    print(f"Saved Stage 5 report to: {report_path}")
else:
    print("Stage 5 returned a status TextOutput — nothing to save (job not complete).")

Saved Stage 5 report to: stage_5_report.md


### Stage 6: Interpretation & Paper Assembly

Takes outputs from all prior stages and produces a publication-style scientific report.

**Input:** Stage-1 hypothesis + Stage-3 experiment design + Stage-4 implementation report + Stage-5 analysis

**Output:** Complete Markdown scientific report (Abstract, Introduction, Methodology, Results, Discussion, Conclusion)

In [3]:
from pathlib import Path

experiment_imp_path = Path("experiment_implementation.md")
experiment_impl_content = experiment_imp_path.read_text(encoding="utf-8")

In [4]:
from pathlib import Path

experiment_aly_path = Path("stage_5_report.md")
experiment_aly_content = experiment_aly_path.read_text(encoding="utf-8")

In [5]:
# Stage 6: Interpretation & Paper Assembly
# Pure LLM agent — takes all prior stage outputs as text, produces a paper.
from akd_ext.agents.closed_loop.cm1 import (
    CM1InterpretationPaperAssemblyAgent,
    CM1InterpretationPaperAssemblyConfig,
)
from akd_ext.agents.closed_loop.stages import (
    InterpretationPaperAssemblyInputSchema,
    InterpretationPaperAssemblyOutputSchema,
)
from IPython.display import Markdown, display

paper_agent = CM1InterpretationPaperAssemblyAgent(CM1InterpretationPaperAssemblyConfig())

paper_result = await paper_agent.arun(
    InterpretationPaperAssemblyInputSchema(
        hypothesis=research_question,
        experiment_design=workflow_spec_content,
        implementation_report=experiment_impl_content,
        experiment_analysis=experiment_aly_content,
    ),
)

if isinstance(paper_result, InterpretationPaperAssemblyOutputSchema):
    print("Paper generated successfully!")
    print("=" * 80)
    display(Markdown(paper_result.report))
else:
    print(paper_result.content)


/Users/rsahoo/Desktop/AKD-EXT-Latest/akd-ext/.venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


Paper generated successfully!


# Environmental Stability Sensitivity of Axisymmetric Tropical Cyclone Intensification in CM1

**Authors:** AI-Augmented Scientific Pipeline (AKD)  
**Date:** 2026-05-14  
**Keywords:** tropical cyclones; environmental stability; CM1; axisymmetric modeling; convection

> *This manuscript was generated with AI assistance and requires researcher validation before publication.*

## Abstract
Environmental thermodynamic stability is expected to modulate tropical cyclone (TC) intensification by altering buoyancy, eyewall convection, and vertical mass flux. This study tests **Hypothesis 3.1**: *TC intensity will increase more rapidly under an unstable environmental temperature profile because enhanced buoyancy strengthens eyewall convection and vertical mass flux, whereas a stable temperature profile suppresses deep convection, leading to weaker storm intensification and reduced peak intensity.* Using CM1 hurricane configurations, three axisymmetric simulations were analyzed: a baseline sounding, a destabilized sounding with a height-dependent cooling reaching **−6 K aloft**, and a stabilized sounding with warming reaching **+6 K aloft**, with surface flux and drag settings held fixed. The unstable perturbation produced the strongest storm, reaching **peak maximum horizontal wind ~108–110 m/s** and **minimum surface pressure ~900 hPa**, compared with the stable perturbation’s **~90–92 m/s** and **~930 hPa**. The baseline was intermediate (**~102–104 m/s**, **~913–915 hPa**). Intensification also occurred earlier in the unstable case (e.g., ~70 m/s by ~50–55 h versus ~80–90 h in the stable case). Convective mass-flux and precipitation diagnostics were larger and more intermittent in the unstable experiment, consistent with the proposed causal chain. Overall, the results **support** Hypothesis 3.1.

## 1. Introduction
Tropical cyclone intensity in idealized numerical experiments is often interpreted through the lens of air–sea disequilibrium and boundary-layer thermodynamics, yet the background thermodynamic profile strongly conditions the ease with which deep convection can be sustained. In particular, the environmental temperature stratification controls parcel buoyancy and inhibition, and therefore the vigor of eyewall updrafts that ventilate the boundary layer, concentrate vorticity, and support warm-core development.

Theory and prior modeling have long emphasized that convective instability and moist thermodynamic efficiency influence TC intensification (e.g., Emanuel, 1986, 1995), while balanced and unbalanced dynamics couple that convection to the vortex (Rotunno & Emanuel, 1987). However, many idealized configurations adopt a single reference sounding, making it difficult to isolate the role of environmental stability from other controls such as surface exchange coefficients, moisture supply, and frictional processes.

This work isolates stability by modifying only the temperature-like column of the input sounding in CM1 while holding moisture, winds, and surface flux/drag parameterizations fixed. The analysis explicitly tests the following hypothesis (verbatim): **Hypothesis 3.1:**
*Tropical cyclone intensity will increase more rapidly under an unstable environmental temperature profile because enhanced buoyancy strengthens eyewall convection and vertical mass flux, whereas a stable temperature profile suppresses deep convection, leading to weaker storm intensification and reduced peak intensity.*

The paper proceeds by describing the experimental design and the mapping between experiments and figure bundles, then presenting results organized around (i) intensity/pressure response and (ii) convective/structural mechanisms that link stability to intensity.

## 2. Experimental Design

### 2.1 Model Configuration
All experiments use the CM1 hurricane test-case family with file-based environmental sounding control (`isnd=7` assumed by the workflow). The primary simulations analyzed here are **axisymmetric** (`axisymm=1` in the reference configuration), which provides a controlled framework for diagnosing mean intensification and structure. Output included time series of intensity, pressure, kinematic structure (e.g., radius of maximum wind, boundary-layer depth), and convection proxies (e.g., maximum vertical velocity, cloud-top height), along with additional integrated diagnostics such as total upward/downward mass flux.

### 2.2 Baseline
The baseline experiment (**EXP_stability_baseline**) uses the unmodified `hurricane_axisymmetric/input_sounding`, with CAPE-family diagnostic outputs enabled (`output_cape=1`, `output_cin=1`, `output_lcl=1`, `output_lfc=1`). Surface flux and drag settings were not modified (no changes to `&param12`), satisfying the causal guardrail that surface exchange parameters remain fixed.

### 2.3 Perturbation Experiments
Two axisymmetric perturbations modify only the temperature-like column (implemented as the `theta` column) of `input_sounding` with a prescribed height-dependent perturbation Δθ(z), while holding moisture and winds fixed.

The **unstable perturbation** (**EXP_stability_001**) applies Δθ(z)=0 K for z≤1 km, ramps to −2 K at 3 km, ramps to −6 K at 12 km, and holds −6 K aloft. The **stable perturbation** (**EXP_stability_002**) applies the same shape with opposite sign (ramps to +2 K at 3 km, +6 K at 12 km, holding +6 K aloft).

Two additional 3D cross-check experiments were defined in the workflow (**EXP_stability_003–004**) but were not identifiable in the Stage-5 figure set analyzed here (Section 2.4). Therefore, this manuscript’s conclusions strictly pertain to the axisymmetric experiments.

### 2.4 Experiment–Figure Mapping
The Stage-5 analysis contains **39 figures** that naturally partition into **three bundles of 13** figures each, where each bundle repeats the same diagnostic set (intensity/pressure/structure time series, convective metrics, mass flux, scatter plots, and peak-metric bars) but with different storm evolution and peak values. Because plot legends show only a single series labeled “output”, the bundles must be mapped using quantitative signatures.

Bundle identification by figure index is:

1) **Bundle 1 (Figures 1–13)**: weaker storm with **peak Vmax ~92 m/s** and **minimum Psfc ~930–940 hPa** (Figure 13 shows Peak Vmax ~92 m/s and Min Psfc in the low-to-mid 930s hPa).  
2) **Bundle 2 (Figures 14–26)**: strongest storm with **peak Vmax ~108–110 m/s** and **minimum Psfc ~900 hPa** (Figure 26 shows Peak Vmax ~108–110 m/s and Min Psfc ~900 hPa).  
3) **Bundle 3 (Figures 27–39)**: intermediate storm with **peak Vmax ~104 m/s** and **minimum Psfc ~913–915 hPa** evident in the intensity/pressure time series (Figure 27 and Figure 33), and a peak-metric bar of **~104 m/s** (Figure 39).

These bundles were mapped to experiments as follows.

Bundle 2 is mapped to the **unstable perturbation (−6 K aloft; EXP_stability_001)** because it exhibits the **largest peak intensity** and **earliest rapid intensification**, matching Hypothesis 3.1’s predicted direction. Bundle 1 is mapped to the **stable perturbation (+6 K aloft; EXP_stability_002)** because it is the **weakest** across peak wind and minimum pressure. Bundle 3 is mapped to the **baseline sounding (EXP_stability_baseline)** because it is consistently **intermediate** between the other two. This mapping is internally consistent across both wind- and pressure-based measures and is therefore treated as high confidence, though it cannot be fully verified without embedded experiment IDs in the plots.

## 3. Results

### 3.1 Intensity and Pressure Response
Hypothesis 3.1 predicts that reduced environmental stability (destabilized temperature profile) should yield faster intensification and higher peak intensity, while increased stability should suppress convection and limit both intensification rate and peak intensity.

The analyzed simulations show a clear, monotonic ordering of storm intensity across stability perturbations. The **unstable perturbation (−6 K)** reaches the highest intensity, with maximum horizontal wind increasing to **~108–110 m/s** late in the simulation and minimum surface pressure falling to **~900–905 hPa**. The **baseline** is slightly weaker, reaching **~102–104 m/s** with minimum surface pressure near **~913–915 hPa** (with some rebound thereafter). The **stable perturbation (+6 K)** is the weakest, peaking near **~90–92 m/s** and only reaching **~930 hPa** at its deepest.

![c4whnhs](https://s.odsi.io/c4whnhs)

**Figure 1:** Unstable perturbation (−6 K aloft): intensity, minimum pressure, RMW, and maximum vertical velocity versus time.

![2k8fxjn](https://s.odsi.io/2k8fxjn)

**Figure 2:** Baseline sounding: intensity, minimum pressure, RMW, and maximum vertical velocity versus time.

![ptdxk9x](https://s.odsi.io/ptdxk9x)

**Figure 3:** Stable perturbation (+6 K aloft): intensity, minimum pressure, RMW, and maximum vertical velocity versus time.

The time evolution also supports a stability control on intensification rate. In the unstable perturbation, rapid intensification begins earlier, with maximum winds climbing from ~25–30 m/s to ~70–75 m/s by approximately **~55 h** (Figure 1). The baseline intensifies strongly as well, reaching ~80–90 m/s by **~70–85 h** before entering a variable mature phase (Figure 2). The stable perturbation shows delayed and weaker deepening, with rapid intensification concentrated around **~60–85 h**, reaching ~70–80 m/s later than the unstable storm (Figure 3). Consistent with these wind evolutions, pressure deficits increase most strongly in the unstable simulation (not shown here as a separate figure because the key signal is already captured by Psfc minima and peak summaries).

To summarize peak outcomes concisely, the peak-metric bar charts (one per experiment bundle) indicate **~108–110 m/s** for the unstable perturbation, **~104 m/s** for the baseline, and **~92 m/s** for the stable perturbation, along with corresponding minima in surface pressure.

![fv5l9i9](https://s.odsi.io/fv5l9i9)

**Figure 4:** Unstable perturbation (−6 K aloft): peak-metric summary (Peak Vmax, Min Psfc, Peak Wmax, Min RMW).

![0u06umz](https://s.odsi.io/0u06umz)

**Figure 5:** Baseline sounding: peak-metric summary (Peak Vmax, Min Psfc, Peak Wmax, Min RMW).

![68aiwqs](https://s.odsi.io/68aiwqs)

**Figure 6:** Stable perturbation (+6 K aloft): peak-metric summary (Peak Vmax, Min Psfc, Peak Wmax, Min RMW).

Across these summaries, the unstable case exceeds the stable case by approximately **~16–18 m/s** in peak maximum wind (about **~18–20%** relative to ~92 m/s) and deepens by roughly **~25–35 hPa** (about **~3%** relative to ~930 hPa). The baseline consistently lies between these endpoints. This pattern directly matches Hypothesis 3.1’s predicted intensity ordering.

### 3.2 Convective and Structural Mechanisms
Hypothesis 3.1 further predicts that the intensity differences should be accompanied by stronger eyewall convection and vertical mass flux in the unstable environment, with suppressed convection in the stable environment.

The time series of convective proxies show that the unstable perturbation sustains larger and more intermittent vertical motions than the stable perturbation. In the unstable simulation, maximum vertical velocity exhibits repeated spikes that reach **~19 m/s** during the core intensification/mature period (Figure 7), whereas the stable simulation’s maximum vertical velocity more typically remains below **~15 m/s** with weaker late-time peaks (Figure 8). Cloud-top height remains deep in both cases once convection is established (generally ~15–18 km; not shown here for both experiments because the cloud-top signals are broadly similar after spin-up), suggesting that the key distinction is less the presence of deep convection and more its intensity and integrated overturning.

![mccwv9s](https://s.odsi.io/mccwv9s)

**Figure 7:** Unstable perturbation (−6 K aloft): convective metrics (including Max w and cloud properties) versus time.

![1ke60l5](https://s.odsi.io/1ke60l5)

**Figure 8:** Stable perturbation (+6 K aloft): convective metrics (including Max w and cloud properties) versus time.

Integrated mass-flux diagnostics provide a more direct measure of overturning. The unstable perturbation shows substantially larger total upward and downward mass flux magnitudes than the stable perturbation, including large-amplitude events (Figure 9) that are absent or much smaller in the stable case (Figure 10). Notably, the unstable mass-flux axes are on a **10^13** scale with spikes approaching **~3.8–3.9×10^13 kg m s⁻¹**, while the stable case peaks on a **10^12** scale near **~2.7–2.8×10^12 kg m s⁻¹**, implying an order-of-magnitude separation in the most extreme overturning episodes. While this comparison is qualitative because the diagnostics’ computation details are not documented in the figure set, the magnitude contrast is consistent with the hypothesized pathway: destabilization enhances buoyancy, which enhances vertical mass flux, which supports stronger intensification.

![l5cqibn](https://s.odsi.io/l5cqibn)

**Figure 9:** Unstable perturbation (−6 K aloft): maximum precipitation rate and total upward/downward mass flux versus time.

![lth7kni](https://s.odsi.io/lth7kni)

**Figure 10:** Stable perturbation (+6 K aloft): maximum precipitation rate and total upward/downward mass flux versus time.

Structural diagnostics are broadly consistent with stronger storms exhibiting smaller radii of maximum wind during the mature phase, but the RMW time series contain discontinuities and step-like behavior in all experiments (e.g., near-zero values or large spikes), implying that the automated RMW diagnostic may become ill-defined during early spin-up or when the vortex is weak. Therefore, RMW-based mechanistic inference is treated cautiously here; the most robust mechanism evidence is instead the consistent ordering of intensity with convective/mass-flux vigor.

### 3.3 Supporting Evidence and Complications
Additional diagnostics support the interpretation that destabilization yields stronger vortex spin-up. Surface vorticity time series (not shown) rise earlier and to larger peaks in the unstable experiment than in the stable experiment, broadly tracking the earlier intensification. However, some convection and structure proxies show discretization (e.g., cloud-base plateaus) and episodic spikes, suggesting threshold-based diagnostic definitions that can complicate quantitative comparison.

Critically, CAPE/CIN/LCL/LFC outputs were enabled by design, but they are not included among the analyzed figures. As a result, the manuscript cannot directly verify that the unstable/stable perturbations produced the expected CAPE/CIN separation in the evolving environment. The causal attribution therefore relies on the guardrailed design (temperature-only sounding modifications) plus the coherent response across intensity and convective overturning metrics.

## 4. Discussion
The results **support Hypothesis 3.1**. The **unstable perturbation (−6 K aloft)** produced a substantially stronger storm than the **stable perturbation (+6 K aloft)**, with **peak maximum horizontal wind ~108–110 m/s versus ~90–92 m/s** (a difference of **~16–18 m/s**, approximately **~18–20%**) and **minimum surface pressure ~900–905 hPa versus ~930–940 hPa** (a deepening of **~25–35 hPa**). The baseline storm was intermediate (**~102–104 m/s**, minimum pressure near **~913–915 hPa**), reinforcing the monotonic intensity ordering with stability.

Mechanistically, the diagnostics are consistent with the intended causal chain. The unstable perturbation exhibits stronger convective intensity proxies (maximum vertical velocity reaching **~19 m/s** versus **≤~15 m/s** in the stable case) and much larger integrated overturning signatures in the mass-flux diagnostics (unstable showing events near **~3.8–3.9×10^13 kg m s⁻¹** versus stable peaks near **~2.7–2.8×10^12 kg m s⁻¹**). These differences align with the interpretation that destabilization increases buoyancy, strengthens eyewall convection and vertical mass flux, and thereby accelerates intensification and raises peak intensity.

Several caveats limit the strength and generality of the conclusion. First, the experiment–figure mapping is inferential because figures lack explicit experiment IDs; nonetheless, the mapping is strongly constrained by the consistent peak-intensity ordering across repeated diagnostic bundles. Second, the simulations analyzed are axisymmetric, which can exaggerate organization and suppress asymmetries that matter for real storms and for 3D convection–vortex interactions. Third, structural metrics (notably RMW) show discontinuities and step-like behavior that may reflect tracking artifacts, especially during early spin-up, making fine-grained structural conclusions uncertain.

In context, the outcome is broadly consistent with potential intensity and moist-convective arguments (Emanuel, 1986, 1995), and with the expectation that the vortex–convection coupling strengthens when buoyancy and convective overturning are more easily realized (Rotunno & Emanuel, 1987). The present results provide a clean single-factor confirmation within this CM1 axisymmetric framework: stability changes imposed solely through the input sounding temperature profile substantially modulate both intensification timing and peak intensity.

## 5. Conclusion
Axisymmetric CM1 experiments in which only the environmental temperature profile was modified demonstrate a strong sensitivity of TC intensification to environmental stability. The destabilized sounding (−6 K aloft) intensified earlier and achieved materially larger peak intensity than the stabilized sounding (+6 K aloft), with **peak winds ~108–110 m/s versus ~90–92 m/s** and **minimum surface pressure ~900–905 hPa versus ~930–940 hPa**. The baseline experiment fell between these endpoints (**~102–104 m/s**, **~913–915 hPa**), producing a monotonic intensity ordering consistent with Hypothesis 3.1.

Convective and overturning diagnostics support a plausible mechanism: destabilization increases the vigor of vertical motions and integrated mass flux, consistent with enhanced eyewall convection supporting stronger spin-up. However, the absence of plotted CAPE/CIN diagnostics and the presence of discontinuities in some structural measures motivate caution in over-interpreting specific structural pathways.

Future work should (i) embed experiment IDs directly into plot legends to remove mapping ambiguity, (ii) explicitly analyze CAPE/CIN/LCL/LFC fields to confirm the thermodynamic separation throughout storm evolution, and (iii) complete and analyze the intended 3D cross-check pair to test whether the stability sensitivity persists under asymmetric convection and more realistic eyewall dynamics.

## Acknowledgments
This manuscript used the CM1 configuration templates `hurricane_axisymmetric` and workflow automation artifacts generated by the AI-Augmented Scientific Pipeline (AKD). The cited literature anchors are acknowledged as conceptual motivation for the tested hypothesis.

In [6]:
# Save the Stage-6 paper
if isinstance(paper_result, InterpretationPaperAssemblyOutputSchema):
    paper_path = Path("stage_6_paper.md")
    paper_path.write_text(paper_result.report, encoding="utf-8")
    print(f"Saved Stage 6 paper to: {paper_path}")
else:
    print("Stage 6 returned a TextOutput — nothing to save.")


Saved Stage 6 paper to: stage_6_paper.md
